# DEEP-SEF — Extended Student Dropout Early-Warning Pipeline
**DEEP-SEF: Turning Early Warning Signs into Actionable Retention Decisions by Combining Three Strongest-Per-Family Tabular Learners — Risk-Factor Impact Analysis at Three Temporal Horizons (t0 / t1 / t2)**

This notebook implements the DEEP-SEF stacking methodology and its risk-factor analysis layer, applied to two datasets — the UCI Predict Students Dropout corpus (primary) and OULAD (cross-dataset) — at three temporal horizons.

1. **At-Risk Student Prediction** — three horizons: pre-semester (t0), after 1st semester (t1), after 2nd semester (t2).
2. **Top Risk-Factor Impact Analysis** per horizon (SHAP-based, grouped into seven risk families: Academic Preparation, Course/Family, Demographics, Financial/Macro, 1st-Semester, 2nd-Semester, Risk Composite).
3. **Base-learner set (dataset-dependent, per §3.4)**: on UCI, the full six-learner base — **LightGBM / CatBoost / XGBoost / ExtraTrees / FT-Transformer / TabPFN**; on OULAD, TabPFN and FT-Transformer are dropped (TabPFN's ~10k-row context limit, FT-Transformer underperforming the gradient-boosted trees) leaving the four-learner **LightGBM / CatBoost / XGBoost / ExtraTrees** subset. Base learners → out-of-fold stacking (best of **LightGBM-meta vs CatBoost-meta**, selected per horizon by CV-AUC) → Temperature Scaling → Mondrian Conformal Prediction (α = 0.10) → Gaussian-mixture Open-World Rejection → SHAP + Counterfactual Explanations.
4. **Multi-task prediction**: Dropout (primary), Graduation, GPA, Attendance mode, Risk agreement.
5. **5-fold Stratified Cross-Validation** for all base learners and both candidate meta-learners, with Optuna (TPE) hyperparameter search per learner per horizon.
6. **Base-learner & ensemble diagnostics**: individual accuracy/precision/recall/F1/AUC per base learner, a base-learner prediction-correlation (diversity) check, and a direct comparison of the stacking ensemble against the best single base learner and a naive mean blend — so it's visible whether stacking, and each included model (including ExtraTrees as the de-randomising counterweight to the boosting learners), is actually earning its place.
7. **Automatic saving** of every CSV and figure that the notebook produces into `./DEEP_SEF_OUTPUT/`.

Runing this notebook will produce a `.zip` at the end containing every artefact produced and will be offered for download(if running in colab).

## Section 0 — Environment Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# CELL 0.1  -  Package Installation (Colab T4 GPU friendly)
# ============================================================
# NOTE: 'ft-transformer' and 'pytorch-tabnet' are NOT installed here.
# The notebook implements its own FT-Transformer as a PyTorch nn.Module later on
# (Section 4.2) and never imports pytorch_tabnet or dice_ml, so those packages
# were dead weight that only added install failures / dependency conflicts.
# Colab's preinstalled torch/torchvision/torchaudio already match the runtime's
# CUDA build, so we do NOT reinstall torch from a pinned cu118 wheel index -
# doing so tends to break `torch.cuda.is_available()` on current Colab images.
!pip -q install --upgrade pip
!pip -q install pandas numpy scikit-learn matplotlib seaborn tqdm joblib
!pip -q install xgboost lightgbm catboost
!pip -q install imbalanced-learn optuna shap
!pip -q install tabpfn

import os, sys, json, time, warnings, math, random, itertools, copy
warnings.filterwarnings('ignore')
print('Python:', sys.version)
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128 | CUDA available: True
Device: Tesla T4


In [ ]:
# ============================================================
# CELL 0.2  -  Constants, Seeding, Output Directory
# ============================================================
import numpy as np
SEED         = 42
SKIP_TABPFN = False   # True = skip TabPFN entirely, always use the LightGBM proxy (no license/token needed)
OUT_DIR      = './DEEP_SEF_OUTPUT'                       # ALL artefacts (CSV + PNGs) saved here
DATA_CSV     = '/content/drive/MyDrive/ML/UCI-StudentDropout.csv'         # mount path inside Colab
N_SPLITS     = 5                                          # 5-fold CV
OPTUNA_TRIALS_BASE = 15                                  # base-learner HPO trials (reduce if slow)
OPTUNA_TRIALS_META = 25                                  # meta-learner HPO trials
RANDOM_SEED  = SEED

for sub in ['', '/csv', '/figures', '/shap', '/models', '/meta']:
    os.makedirs(OUT_DIR + sub, exist_ok=True)

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
try:
    torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)
except Exception:
    pass
print(f'Output directory: {OUT_DIR}')
print(f'5-fold CV  |  Base HPO trials: {OPTUNA_TRIALS_BASE}  |  Meta HPO trials: {OPTUNA_TRIALS_META}')
print(f'Seed = {SEED}')

Output directory: ./DEEP_SEF_OUTPUT
5-fold CV  |  Base HPO trials: 15  |  Meta HPO trials: 25
Seed = 42


### 0.3  Upload / mount the dataset
If you uploaded the file to `/content/`, this code loads it automatically. Otherwise place the CSV at `DATA_CSV`.

In [ ]:
# ============================================================
# CELL 0.3  -  Load UCI student-dropout CSV
# ============================================================
import pandas as pd, numpy as np

df_raw = pd.read_csv(DATA_CSV, sep=';')
df_raw.columns = [c.strip() for c in df_raw.columns]
print('Raw shape:', df_raw.shape)
print('Columns :', list(df_raw.columns))
print('Target distribution:')
print(df_raw['Target'].value_counts())
df_raw.to_csv(f'{OUT_DIR}/csv/00_raw_dataset_snapshot.csv', index=False)
print(f'Saved -> {OUT_DIR}/csv/00_raw_dataset_snapshot.csv')

Raw shape: (4424, 37)
Columns : ['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate'

## Section 1 — Data Preprocessing (Binary Subset)
Following DEEP-SEF: drop `Enrolled`, encode `Graduate→0`, `Dropout→1`.

In [ ]:
# ============================================================
# CELL 1.1  -  Drop Enrolled rows, encode Target
# ============================================================
df = df_raw[df_raw['Target'].isin(['Graduate', 'Dropout'])].copy()
df['Target'] = (df['Target'] == 'Dropout').astype(int)
print('Binary working set shape:', df.shape, '| dropout rate:', df['Target'].mean().round(4))
df.to_csv(f'{OUT_DIR}/csv/01_binary_dataset.csv', index=False)

RENAME_MAP = {
    'Nacionality': 'Nationality',
    'Daytime/evening attendance': 'Attendance',
    'Age at enrollment': 'Age',
    'Previous qualification (grade)': 'PreviousGrade',
    "Mother's qualification": 'MotherQual',
    "Father's qualification": 'FatherQual',
    "Mother's occupation":  'MotherOcc',
    "Father's occupation":  'FatherOcc',
    'Curricular units 1st sem (credited)':        's1_credited',
    'Curricular units 1st sem (enrolled)':        's1_enrolled',
    'Curricular units 1st sem (evaluations)':     's1_eval',
    'Curricular units 1st sem (approved)':        's1_approved',
    'Curricular units 1st sem (grade)':           's1_grade',
    'Curricular units 1st sem (without evaluations)': 's1_wo_eval',
    'Curricular units 2nd sem (credited)':        's2_credited',
    'Curricular units 2nd sem (enrolled)':        's2_enrolled',
    'Curricular units 2nd sem (evaluations)':     's2_eval',
    'Curricular units 2nd sem (approved)':        's2_approved',
    'Curricular units 2nd sem (grade)':           's2_grade',
    'Curricular units 2nd sem (without evaluations)': 's2_wo_eval',
    'Unemployment rate': 'UnempRate',
    'Inflation rate':    'Inflation',
    'GDP':               'GDP',
    'Admission grade':   'AdmissionGrade',
    'Application mode':  'ApplicationMode',
    'Application order': 'ApplicationOrder',
    'Previous qualification':          'PrevQual',
    'Educational special needs':       'SpecialNeeds',
    'Scholarship holder':              'Scholarship',
    'Tuition fees up to date':         'TuitionUpToDate',
    'Marital status':                  'MaritalStatus',
}
df = df.rename(columns=RENAME_MAP)
print('Renamed columns sample:', list(df.columns)[:10], '...')
df.to_csv(f'{OUT_DIR}/csv/02_renamed_dataset.csv', index=False)
print('Saved -> 02_renamed_dataset.csv')

Binary working set shape: (3630, 37) | dropout rate: 0.3915
Renamed columns sample: ['MaritalStatus', 'ApplicationMode', 'ApplicationOrder', 'Course', 'Attendance', 'PrevQual', 'PreviousGrade', 'Nationality', 'MotherQual', 'FatherQual'] ...
Saved -> 02_renamed_dataset.csv


## Section 2 — Feature Engineering (Five-Group Schema)
Mirrors DEEP-SEF's 36→79 expansion; emits only the engineered groups available at each horizon.

In [ ]:
# ============================================================
# CELL 2.1  -  Five-Group feature engineering
# ============================================================
def safe_div(a, b):
    return np.where(b == 0, 0.0, a / np.where(b == 0, 1, b))

def engineer_features(df_in: pd.DataFrame, horizon: str) -> pd.DataFrame:
    """
    Returns engineered features for a given horizon.
    horizon ∈ {'t0', 't1', 't2'}
        t0 — only demographic + socioeconomic + macro features (Pre-Semester).
        t1 — t0 features + 1st-semester curricular aggregates (After 1st Semester).
        t2 — t0 + t1 + 2nd-semester curricular aggregates (After 2nd Semester = full).
    """
    df = df_in.copy()

    # ----- G2 risk composite scores (always available) -----
    df['financial_risk']    = (df['Debtor'].astype(int) + (1 - df['TuitionUpToDate'].astype(int))).clip(0, 2)
    df['vulnerability_idx']  = df['financial_risk'] + df['Scholarship'].astype(int) + df['Displaced'].astype(int)
    df['academic_distress']  = 0          # placeholder, will be filled if curricular data exists
    df['total_failures']     = 0

    # ----- G5 contextual interactions (always available) -----
    df['age_x_scholarship']     = df['Age'] * df['Scholarship']
    df['age_x_financial_risk']  = df['Age'] * df['financial_risk']
    df['gdp_x_debtor']          = df['GDP']  * df['Debtor']
    df['unemp_x_age']           = df['UnempRate'] * df['Age']
    df['gdp_x_tuition']         = df['GDP']  * df['TuitionUpToDate']
    df['inflation_x_debtor']    = df['Inflation'] * df['Debtor']
    df['admission_vs_prevgrade'] = df['AdmissionGrade'] - df['PreviousGrade']
    df['parent_qual_avg']       = (df['MotherQual'] + df['FatherQual']) / 2.0
    df['parent_occ_avg']        = (df['MotherOcc'] + df['FatherOcc']) / 2.0

    if horizon in ('t1', 't2'):
        # ----- G1 academic momentum (after 1st semester) -----
        s1_eval    = df['s1_eval'].to_numpy()
        s1_approved= df['s1_approved'].to_numpy()
        s1_enrolled= df['s1_enrolled'].to_numpy()
        s1_credited= df['s1_credited'].to_numpy()
        s1_grade   = df['s1_grade'].to_numpy()
        s1_wo      = df['s1_wo_eval'].to_numpy()

        df['s1_approval_rate']  = safe_div(s1_approved, s1_enrolled)
        df['s1_eval_rate']      = safe_div(s1_eval,    s1_enrolled)
        df['s1_credit_rate']    = safe_div(s1_credited,s1_enrolled)
        df['s1_pass_efficiency']= safe_div(s1_approved, np.maximum(s1_eval,1))
        df['s1_eval_gap']       = s1_enrolled - s1_eval
        df['s1_failed']         = (s1_enrolled - s1_approved).clip(min=0)
        df['total_failures']    = df['total_failures'] + df['s1_failed']
        df['low_grade_flag_s1'] = ((s1_grade > 0) & (s1_grade < 10)).astype(int)
        df['zero_grade_s1']     = (s1_grade == 0).astype(int)
        df['academic_distress'] = df['s1_failed'] + df['low_grade_flag_s1'] + df['zero_grade_s1']

    if horizon == 't2':
        s2_eval    = df['s2_eval'].to_numpy()
        s2_approved= df['s2_approved'].to_numpy()
        s2_enrolled= df['s2_enrolled'].to_numpy()
        s2_credited= df['s2_credited'].to_numpy()
        s2_grade   = df['s2_grade'].to_numpy()
        s2_wo      = df['s2_wo_eval'].to_numpy()

        df['s2_approval_rate']  = safe_div(s2_approved, s2_enrolled)
        df['s2_eval_rate']      = safe_div(s2_eval,    s2_enrolled)
        df['s2_credit_rate']    = safe_div(s2_credited,s2_enrolled)
        df['s2_pass_efficiency']= safe_div(s2_approved, np.maximum(s2_eval,1))
        df['s2_eval_gap']       = s2_enrolled - s2_eval
        df['s2_failed']         = (s2_enrolled - s2_approved).clip(min=0)
        df['s1_to_s2_grade_delta'] = df['s2_grade'] - df['s1_grade']
        df['s1_to_s2_approved_delta'] = df['s2_approved'] - df['s1_approved']
        df['avg_grade']         = (df['s1_grade'] + df['s2_grade']) / 2.0
        df['cum_approved']      = df['s1_approved'] + df['s2_approved']
        df['cum_failed']        = df['s1_failed']   + df['s2_failed']
        df['total_failures']    = df['total_failures'] + df['s2_failed']
        df['low_grade_flag_s2'] = ((s2_grade > 0) & (s2_grade < 10)).astype(int)
        df['zero_grade_s2']     = (s2_grade == 0).astype(int)
        df['academic_distress'] = df['academic_distress'] + df['s2_failed'] + df['low_grade_flag_s2'] + df['zero_grade_s2']
        df['momentum_score']    = df['s1_to_s2_grade_delta'] - df['s1_to_s2_approved_delta']
        df['cum_efficiency']    = safe_div(df['cum_approved'].to_numpy(), df['cum_approved'].to_numpy() + df['cum_failed'].to_numpy())
        df['grade_per_approved']= safe_div(df['avg_grade'].to_numpy(), np.maximum(df['cum_approved'].to_numpy(),1))

    # ----- G3 academic efficiency (when data available) -----
    if horizon in ('t1','t2'):
        df['eval_participation_gap'] = df['s1_eval_gap']
    if horizon == 't2':
        df['eval_participation_gap'] = df['s1_eval_gap'] + df['s2_eval_gap']

    df['academic_distress']    = df['academic_distress'].fillna(0)
    df['financial_risk']       = df['financial_risk'].clip(lower=0)
    df['vulnerability_idx']    = df['vulnerability_idx'].clip(lower=0)
    df['total_failures']       = df['total_failures'].clip(lower=0)
    df.replace([np.inf, -np.inf], 0, inplace=True)
    return df

def get_horizon_columns(df_in: pd.DataFrame, horizon: str):
    """Return the predictor columns available at the given horizon (numeric-only for safety)."""
    drop = ['Target']
    s2_cols = [c for c in df_in.columns if c.startswith('s2_')]
    if horizon == 't0':
        drop += s2_cols + ['s1_grade','s1_approved','s1_enrolled','s1_eval','s1_credited','s1_wo_eval',
                           's1_approval_rate','s1_eval_rate','s1_credit_rate','s1_pass_efficiency',
                           's1_eval_gap','s1_failed','low_grade_flag_s1','zero_grade_s1']
    elif horizon == 't1':
        drop += s2_cols
    return [c for c in df_in.columns if c not in drop]

print('Engineered feature schema ready.')

Engineered feature schema ready.


In [ ]:
# ============================================================
# CELL 2.2  -  Build & persist per-horizon datasets
# ============================================================
horizons = ['t0', 't1', 't2']
horizon_desc = {
    't0':'pre_semester',
    't1':'after_sem1',
    't2':'after_sem2'
}
datasets = {}
for h in horizons:
    eng  = engineer_features(df, h)
    cols = get_horizon_columns(eng, h)
    X = eng[cols].astype(np.float32)
    y = eng['Target'].astype(int)
    datasets[h] = {'X': X, 'y': y, 'columns': cols}
    eng[cols + ['Target']].to_csv(f'{OUT_DIR}/csv/03_features_{h}_{horizon_desc[h]}.csv', index=False)
    print(f'{h} ({horizon_desc[h]:>13})  |  features: {len(cols)}  |  dropout rate: {y.mean():.4f}')

feat_counts = pd.DataFrame([
    {'horizon': h, 'description': horizon_desc[h],
     'n_features': len(datasets[h]['columns']),
     'n_rows':     len(datasets[h]['X']),
     'dropout_rate': datasets[h]['y'].mean()}
    for h in horizons
])
feat_counts.to_csv(f'{OUT_DIR}/csv/04_feature_counts_per_horizon.csv', index=False)
print()
print(feat_counts)

t0 ( pre_semester)  |  features: 37  |  dropout rate: 0.3915
t1 (   after_sem1)  |  features: 52  |  dropout rate: 0.3915
t2 (   after_sem2)  |  features: 74  |  dropout rate: 0.3915

  horizon   description  n_features  n_rows  dropout_rate
0      t0  pre_semester          37    3630       0.39146
1      t1    after_sem1          52    3630       0.39146
2      t2    after_sem2          74    3630       0.39146


## Section 3 — Train/Test Split, Borderline-SMOTE + Tomek Resampling (per fold)

In [ ]:
# ============================================================
# CELL 3.1  -  Stratified 80/20 split + 5-fold CV definition
# ============================================================
from sklearn.model_selection import train_test_split, StratifiedKFold
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import BorderlineSMOTE
from collections import Counter

def split_and_make_folds(X, y, horizon, n_splits=N_SPLITS, seed=SEED):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for tr_idx, va_idx in cv.split(X_tr, y_tr):
        Xf_tr, yf_tr = X_tr.iloc[tr_idx].copy(), y_tr.iloc[tr_idx].copy()
        Xf_va, yf_va = X_tr.iloc[va_idx].copy(), y_tr.iloc[va_idx].copy()
        # Borderline-SMOTE Type-2 + Tomek Links only on training fold (no leakage)
        try:
            bsm = BorderlineSMOTE(kind='borderline-2', random_state=seed, sampling_strategy=0.85)
            Xf_tr_r, yf_tr_r = bsm.fit_resample(Xf_tr, yf_tr)
            st = SMOTETomek(tomek=SMOTETomek(sampling_strategy='majority').tomek, random_state=seed)
            Xf_tr_r, yf_tr_r = st.fit_resample(Xf_tr_r, yf_tr_r)
        except Exception as e:
            print(f'[WARN] Resampling fallback ({e}): using original fold.')
            Xf_tr_r, yf_tr_r = Xf_tr.copy(), yf_tr.copy()
        # Ensure numeric
        Xf_tr_r = pd.DataFrame(Xf_tr_r, columns=Xf_tr.columns).astype(np.float32)
        Xf_va   = Xf_va.astype(np.float32)
        folds.append({'X_tr': Xf_tr_r, 'y_tr': pd.Series(yf_tr_r),
                     'X_va': Xf_va,       'y_va': yf_va})
    return X_tr.astype(np.float32), X_te.astype(np.float32), y_tr, y_te, folds

splits = {}
for h in horizons:
    X_tr, X_te, y_tr, y_te, folds = split_and_make_folds(datasets[h]['X'], datasets[h]['y'], h)
    splits[h] = {'X_tr': X_tr, 'X_te': X_te, 'y_tr': y_tr, 'y_te': y_te, 'folds': folds}
    print(f'{h} | train {X_tr.shape} | test {X_te.shape} | folds={len(folds)} | class balance(train raw): {Counter(y_tr)}')
    sample_after = Counter(splits[h]['folds'][0]['y_tr'].tolist())
    print(f'    sample fold after resampling: {sample_after}\n')

# Persist a small summary
summary_split = []
for h in horizons:
    s = splits[h]
    n0 = (s['y_tr']==0).sum(); n1 = (s['y_tr']==1).sum()
    summary_split.append({'horizon':h, 'train_rows':len(s['y_tr']),
                           'test_rows':len(s['y_te']),
                           'features':s['X_tr'].shape[1],
                           'train_grad':int(n0),'train_drop':int(n1)})
pd.DataFrame(summary_split).to_csv(f'{OUT_DIR}/csv/05_split_summary.csv', index=False)
print('Saved -> 05_split_summary.csv')

t0 | train (2904, 37) | test (726, 37) | folds=5 | class balance(train raw): Counter({0: 1767, 1: 1137})
    sample fold after resampling: Counter({1: 1315, 0: 1315})

t1 | train (2904, 52) | test (726, 52) | folds=5 | class balance(train raw): Counter({0: 1767, 1: 1137})
    sample fold after resampling: Counter({1: 1326, 0: 1326})

t2 | train (2904, 74) | test (726, 74) | folds=5 | class balance(train raw): Counter({0: 1767, 1: 1137})
    sample fold after resampling: Counter({1: 1356, 0: 1356})

Saved -> 05_split_summary.csv


## Section 4 — Optuna-tuned Base Learners (LightGBM, CatBoost, XGBoost, ExtraTrees, FT-Transformer, TabPFN)

Six base learners are tuned here: four tree ensembles (LightGBM, CatBoost, XGBoost, ExtraTrees), a deep tabular attention model (FT-Transformer, Section 4.2-4.3), and a prior-fitted tabular network (TabPFN, Section 4.4, with a LightGBM proxy fallback).

An MLP/ANN and Gaussian Naive Bayes were tried in an earlier version of this pipeline and dropped: on the diagnostics in Section 5.3 neither ever beat the weakest tree model on this dataset, so they only added training time and correlated noise to the stacking matrix without buying diversity. **ExtraTrees replaces them** instead — on this exact UCI student-dropout/retention dataset, published benchmarks report Extra Trees as the strongest individual model in head-to-head comparisons against Random Forest, Gradient Boosting and Logistic Regression (e.g. Extra Trees reaching ~87% accuracy / 0.96 macro-AUC on this dataset, beating RF/GB/LR — *Enhancing Student Retention in HEIs: A Machine Learning Approach*, Electronics 2026), and Optuna-tuned LightGBM/CatBoost consistently outperform traditional classifiers on comparable dropout-prediction benchmarks (*Supervised ML algorithms for predicting student dropout*, Discover AI 2024). ExtraTrees' extremely-randomised split selection also gives genuinely different errors from LightGBM/CatBoost/XGBoost's gradient-guided splits, so it earns a place in the stacking matrix on diversity grounds too — unlike MLP/NB it does so without a standalone-accuracy penalty.

In [ ]:
# ============================================================
# CELL 4.1  -  Imports & helper utilities for base learners
# ============================================================
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score, precision_score, recall_score,
                             confusion_matrix, brier_score_loss, log_loss, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.pipeline import make_pipeline
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device} for FT-Transformer / TabPFN / TabNet models.')

# ------------- Search-space wrappers (per horizon) -------------
def lgb_search_space(trial):
    return {
        'objective': 'binary', 'random_state': SEED, 'verbose': -1,
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 200),
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'min_child_samples':trial.suggest_int('min_child_samples', 10, 80),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }

def cat_search_space(trial):
    return {
        'iterations':  trial.suggest_int('iterations', 200, 800),
        'learning_rate':trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'depth':        trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':  trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'random_seed':  SEED, 'verbose': False,
    }

def xgb_search_space(trial):
    return {
        'n_estimators':   trial.suggest_int('n_estimators', 200, 800),
        'learning_rate':  trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'max_depth':      trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0),
        'subsample':      trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':      trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda':     trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'eval_metric':    'logloss',
        'random_state':   SEED,
        'tree_method':    'hist',
        'device':         device if device=='cuda' else 'cpu',
    }

def et_search_space(trial):
    # ExtraTrees (extremely randomised trees): splits are chosen at random
    # thresholds rather than the gradient-optimal ones LightGBM/CatBoost/XGBoost
    # pick, so its errors are genuinely decorrelated from the other three tree
    # ensembles -- and it is independently reported as the strongest single
    # model on this exact dataset in the literature (see Section 4 markdown).
    return {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'max_depth':        trial.suggest_int('max_depth', 6, 24),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'min_samples_split':trial.suggest_int('min_samples_split', 2, 20),
        'max_features':     trial.suggest_float('max_features', 0.4, 1.0),
        'random_state':     SEED,
        'n_jobs':           -1,
    }

# ------------ Cross-validated objective ------------
def cv_auc_objective(make_estimator, params, folds, scale=False):
    """Return mean ROC-AUC across folds."""
    aucs = []
    for fold in folds:
        Xtr, ytr = fold['X_tr'], fold['y_tr']
        Xva, yva = fold['X_va'], fold['y_va']
        if scale:
            sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xva = sc.transform(Xva)
        try:
            mdl = make_estimator(params).fit(Xtr, ytr)
            p   = mdl.predict_proba(Xva)[:,1]
        except Exception:
            return 0.0
        aucs.append(roc_auc_score(yva, p))
    return float(np.mean(aucs))

def optimize(horizon, splits_h, optuna_trials=OPTUNA_TRIALS_BASE):
    folds = splits_h['folds']
    out = {}
    for name, mk, ss, scale in [
        ('LightGBM', lambda p: lgb.LGBMClassifier(**p),         lgb_search_space,  False),
        ('CatBoost', lambda p: CatBoostClassifier(**p),         cat_search_space,   False),
        ('XGBoost',   lambda p: xgb.XGBClassifier(**p),          xgb_search_space,  False),
        ('ExtraTrees',lambda p: ExtraTreesClassifier(**p),         et_search_space,   False),
    ]:
        study = optuna.create_study(direction='maximize',
                                    sampler=TPESampler(seed=SEED),
                                    study_name=f'{name}_{horizon}')
        def _obj(trial):
            params = ss(trial)
            return cv_auc_objective(mk, params, folds, scale=scale)
        study.optimize(_obj, n_trials=optuna_trials, show_progress_bar=False)
        out[name] = {'best_value': study.best_value, 'best_params': study.best_params}
        print(f'   [{horizon}] {name}: best CV-AUC = {study.best_value:.4f}')
    return out

hpo_results = {}
for h in horizons:
    print(f'\n== HPO for horizon {h} ({horizon_desc[h]}) ==')
    hpo_results[h] = optimize(h, splits[h], optuna_trials=OPTUNA_TRIALS_BASE)
    pd.DataFrame([
        {'horizon': h, 'model': m, 'cv_auc': r['best_value']}
        for m, r in hpo_results[h].items()
    ]).to_csv(f'{OUT_DIR}/csv/06_hpo_results_{h}.csv', index=False)
    # Save best params
    with open(f'{OUT_DIR}/meta/06_hpo_best_params_{h}.json', 'w') as f:
        json.dump(hpo_results[h], f, indent=2, default=str)
print('HPO done for all three horizons.')


Using device: cuda for FT-Transformer / TabPFN / TabNet models.

== HPO for horizon t0 (pre_semester) ==
   [t0] LightGBM: best CV-AUC = 0.8476
   [t0] CatBoost: best CV-AUC = 0.8447
   [t0] XGBoost: best CV-AUC = 0.8451
   [t0] ExtraTrees: best CV-AUC = 0.8269

== HPO for horizon t1 (after_sem1) ==
   [t1] LightGBM: best CV-AUC = 0.9370
   [t1] CatBoost: best CV-AUC = 0.9353
   [t1] XGBoost: best CV-AUC = 0.9352
   [t1] ExtraTrees: best CV-AUC = 0.9310

== HPO for horizon t2 (after_sem2) ==
   [t2] LightGBM: best CV-AUC = 0.9520
   [t2] CatBoost: best CV-AUC = 0.9531
   [t2] XGBoost: best CV-AUC = 0.9529
   [t2] ExtraTrees: best CV-AUC = 0.9504
HPO done for all three horizons.


### 4.2  FT-Transformer (deep tabular learner)
We'll implement a compact FT-Transformer in PyTorch. Training per fold keeps within Colab T4 limits.

In [ ]:
# ============================================================
# CELL 4.2  -  FT-Transformer (PyTorch)
# ============================================================
import torch.nn as nn

class FTTransformer(nn.Module):
    def __init__(self, n_features, d_token=32, n_blocks=3, n_heads=4, dropout=0.1):
        super().__init__()
        self.tokenizer = nn.Linear(1, d_token)
        self.feature_emb = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.cls         = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)
        encoder_layer    = nn.TransformerEncoderLayer(d_model=d_token, nhead=n_heads,
                                                    dim_feedforward=4*d_token, dropout=dropout,
                                                    batch_first=True, activation='gelu')
        self.encoder    = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.norm       = nn.LayerNorm(d_token)
        self.head       = nn.Sequential(nn.Linear(d_token, d_token), nn.GELU(),
                                        nn.Dropout(dropout), nn.Linear(d_token, 1))
        self.n_features = n_features
    def forward(self, x):
        # x: [B, F] -> tokenise each feature
        B, F = x.shape
        tok = self.tokenizer(x.unsqueeze(-1))              # [B, F, d]
        tok = tok + self.feature_emb.unsqueeze(0)          # add feature embedding
        cls = self.cls.expand(B, -1, -1)                   # [B, 1, d]
        tok = torch.cat([cls, tok], dim=1)                 # [B, F+1, d]
        h   = self.encoder(tok)
        cls_out = self.norm(h[:,0,:])
        return self.head(cls_out).squeeze(-1)              # logits

def train_ftt(Xtr, ytr, Xva, yva, n_epochs=40, lr=1e-3, batch_size=256, d_token=32, n_blocks=3, n_heads=4, dropout=0.1, verbose=False):
    sc = StandardScaler().fit(Xtr)
    Xtr_t = torch.tensor(sc.transform(Xtr), dtype=torch.float32, device=device)
    Xva_t = torch.tensor(sc.transform(Xva), dtype=torch.float32, device=device)
    ytr_t = torch.tensor(ytr.to_numpy() if hasattr(ytr,'to_numpy') else ytr,
                         dtype=torch.float32, device=device)
    yva_a = yva.to_numpy() if hasattr(yva,'to_numpy') else yva
    model = FTTransformer(n_features=Xtr.shape[1], d_token=d_token, n_blocks=n_blocks,
                           n_heads=n_heads, dropout=dropout).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    bce   = nn.BCEWithLogitsLoss()
    n = Xtr_t.size(0)
    best_auc, best_state = -1, None
    for ep in range(n_epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            opt.zero_grad()
            logit = model(Xtr_t[idx])
            loss  = bce(logit, ytr_t[idx])
            loss.backward(); opt.step()
        sched.step()
        # Eval
        model.eval()
        with torch.no_grad():
            p = torch.sigmoid(model(Xva_t)).cpu().numpy()
        auc = roc_auc_score(yva_a, p)
        if auc > best_auc:
            best_auc, best_state = auc, copy.deepcopy(model.state_dict())
        if verbose and ep % 10 == 0:
            print(f'    ep {ep:03d} | val AUC {auc:.4f}')
    model.load_state_dict(best_state)
    return model, sc, best_auc

def predict_ftt(model, sc, X):
    model.eval()
    Xt = torch.tensor(sc.transform(X), dtype=torch.float32, device=device)
    with torch.no_grad():
        return torch.sigmoid(model(Xt)).cpu().numpy()
print('FT-Transformer utilities ready.')

FT-Transformer utilities ready.


In [ ]:
# ============================================================
# CELL 4.3  -  Train FT-Transformer across folds with HPO
# ============================================================
ftt_results = {}
for h in horizons:
    print(f'\n== FT-Transformer training (horizon {h}) ==')
    folds = splits[h]['folds']
    best_auc = -1
    best_cfg  = None
    trial_log = []
    for trial_idx, cfg in enumerate([
            {'d_token':16,'n_blocks':2,'n_heads':2,'dropout':0.1,'lr':1e-3,'n_epochs':25,'batch_size':256},
            {'d_token':32,'n_blocks':3,'n_heads':4,'dropout':0.1,'lr':5e-4,'n_epochs':30,'batch_size':256},
            {'d_token':24,'n_blocks':2,'n_heads':4,'dropout':0.15,'lr':1e-3,'n_epochs':25,'batch_size':128},
    ]):
        aucs = []
        for fi, fold in enumerate(folds):
            model, sc, auc = train_ftt(fold['X_tr'], fold['y_tr'], fold['X_va'], fold['y_va'], **cfg)
            aucs.append(auc)
        m = float(np.mean(aucs))
        trial_log.append({'trial': trial_idx, 'cfg': cfg, 'cv_auc': m})
        print(f'   trial {trial_idx} -> CV-AUC {m:.4f}')
        if m > best_auc:
            best_auc = m; best_cfg = cfg
    ftt_results[h] = {'best_cfg': best_cfg, 'cv_auc': best_auc, 'trials': trial_log}
    pd.DataFrame([{'trial': t['trial'], 'cfg': str(t['cfg']), 'cv_auc': t['cv_auc']} for t in trial_log])\
        .to_csv(f'{OUT_DIR}/csv/07_fttransformer_hpo_{h}.csv', index=False)
    print(f'   ===> best CV-AUC {best_auc:.4f}  cfg={best_cfg}')
print('\nFT-Transformer HPO done.')


== FT-Transformer training (horizon t0) ==
   trial 0 -> CV-AUC 0.7558
   trial 1 -> CV-AUC 0.7532
   trial 2 -> CV-AUC 0.8099
   ===> best CV-AUC 0.8099  cfg={'d_token': 24, 'n_blocks': 2, 'n_heads': 4, 'dropout': 0.15, 'lr': 0.001, 'n_epochs': 25, 'batch_size': 128}

== FT-Transformer training (horizon t1) ==
   trial 0 -> CV-AUC 0.9065
   trial 1 -> CV-AUC 0.9100
   trial 2 -> CV-AUC 0.9218
   ===> best CV-AUC 0.9218  cfg={'d_token': 24, 'n_blocks': 2, 'n_heads': 4, 'dropout': 0.15, 'lr': 0.001, 'n_epochs': 25, 'batch_size': 128}

== FT-Transformer training (horizon t2) ==
   trial 0 -> CV-AUC 0.9143
   trial 1 -> CV-AUC 0.9172
   trial 2 -> CV-AUC 0.9406
   ===> best CV-AUC 0.9406  cfg={'d_token': 24, 'n_blocks': 2, 'n_heads': 4, 'dropout': 0.15, 'lr': 0.001, 'n_epochs': 25, 'batch_size': 128}

FT-Transformer HPO done.


In [ ]:
import os, getpass

if SKIP_TABPFN:
    print('SKIP_TABPFN=True — skipping token prompt, TabPFN will not be used.')
elif not os.environ.get('TABPFN_TOKEN'):
    try:
        token = getpass.getpass('Enter your TABPFN_TOKEN: ')
    except Exception:
        token = ''
    if token.strip():
        os.environ['TABPFN_TOKEN'] = token.strip()
        print('TABPFN_TOKEN set for this session.')
    else:
        print('No token provided — TabPFN will fall back to the LightGBM proxy automatically.')
else:
    print('TABPFN_TOKEN already set in environment.')

tabpfn_sk_pzgecc1_yOXjXzn4suTAEzvSwEfmiMGqvm5YwEXEFL4··········
TABPFN_TOKEN set for this session.


### 4.4  TabPFN — conditional inference prior-fitted network (small tabular model)

In [ ]:
# ============================================================
# CELL 4.4  -  TabPFN baseline (with graceful fallback)
# ============================================================
# USE_TABPFN = True
USE_TABPFN = not SKIP_TABPFN
try:
    from tabpfn import TabPFNClassifier
    _ = TabPFNClassifier(device=device, n_estimators=2)
    print(f'TabPFN available on {device}.')
    TABPFN_DEVICE = device
    TABPFN_NE = 4
except Exception as e:
    USE_TABPFN = False
    print(f'TabPFN unavailable ({e}); using a calibrated LightGBM proxy for the 4th base learner.')

tabpfn_results = {}
if USE_TABPFN:
    for h in horizons:
        folds = splits[h]['folds']
        oof, val_aucs = [], []
        for fi, fold in enumerate(folds):
            Xtr, ytr = fold['X_tr'].to_numpy(), fold['y_tr'].to_numpy()
            Xva, yva = fold['X_va'].to_numpy(), fold['y_va'].to_numpy()
            try:
                m = TabPFNClassifier(device=TABPFN_DEVICE, n_estimators=TABPFN_NE,
                                     random_state=SEED, ignore_pretraining_limits=True)
                m.fit(Xtr, ytr)
                p = m.predict_proba(Xva)[:,1]
            except Exception as e:
                print(f'    TabPFN fold {fi} failed -> falling back to LightGBM ({e})')
                m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=64, random_state=SEED, verbose=-1)
                m.fit(Xtr, ytr); p = m.predict_proba(Xva)[:,1]
            oof.append(p); val_aucs.append(roc_auc_score(yva, p))
        tabpfn_results[h] = {'cv_auc': float(np.mean(val_aucs)), 'folds_auc': val_aucs}
        print(f'  TabPFN | {h} | CV-AUC {tabpfn_results[h]["cv_auc"]:.4f}')
        pd.DataFrame({'fold': range(len(val_aucs)), 'val_auc': val_aucs})\
            .to_csv(f'{OUT_DIR}/csv/08_tabpfn_cv_{h}.csv', index=False)
else:
    for h in horizons:
        tabpfn_results[h] = {'cv_auc': float('nan'), 'folds_auc': []}
print('TabPFN results ready.')

TabPFN available on cuda.


tabpfn-v3-classifier-v3_default.ckpt:   0%|          | 0.00/213M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

  TabPFN | t0 | CV-AUC 0.8469
  TabPFN | t1 | CV-AUC 0.9360
  TabPFN | t2 | CV-AUC 0.9530
TabPFN results ready.


## Section 5 — Generate Out-Of-Fold Predictions & Final-Base Predictions on Test

OOF and test-set predictions are generated for all six base learners (LightGBM, CatBoost, XGBoost, ExtraTrees, FT-Transformer, TabPFN). Section 5.3 then scores each base learner individually (Accuracy/Precision/Recall/F1/AUC) so their standalone performance is visible before they're combined in the stacking ensemble.

In [ ]:
# ============================================================
# CELL 5.1  -  Build OOF + test predictions for all base learners
# ============================================================
def get_best_estimator(name, params):
    """Instantiate a base learner from a (possibly overridden) parameter dict.

    The dict produced by lgb_search_space / cat_search_space / xgb_search_space /
    mlp_search_space / nb_search_space already includes the fixed parameters
    (objective, random_state, verbose, ...). We therefore spread it with
    ``**params`` and add only the *additional* knobs that were defined outside
    the search-space wrappers (so we do not duplicate any keyword).
    """
    if name == 'LightGBM':
        # objective / random_state / verbose are already in params
        return lgb.LGBMClassifier(**params)
    if name == 'CatBoost':
        # random_seed / verbose are already in params
        return CatBoostClassifier(**params)
    if name == 'XGBoost':
        # eval_metric / random_state / tree_method / device are already in params.
        # We still want a safe histogram tree-method fallback for CPU-only Colab.
        p = dict(params)
        p.setdefault('tree_method', 'hist')
        return xgb.XGBClassifier(**p)
    if name == 'ExtraTrees':
        return ExtraTreesClassifier(**params)
    raise ValueError(name)

def ft_predict_best(Xtr, ytr, Xva, yva, Xtest, cfg):
    model, sc, _ = train_ftt(Xtr, ytr, Xva, yva, **cfg)
    return predict_ftt(model, sc, Xva), predict_ftt(model, sc, Xtest)

def predict_test_with_best_params(name, params, X_tr, y_tr, X_te):
    m = get_best_estimator(name, params)
    m.fit(X_tr, y_tr)
    return m.predict_proba(X_te)[:,1]

def get_oof_for_learner(name, params, folds):
    oof, val_true = [], []
    for fold in folds:
        m = get_best_estimator(name, params)
        m.fit(fold['X_tr'], fold['y_tr'])
        p = m.predict_proba(fold['X_va'])[:,1]
        oof.append(p); val_true.append(fold['y_va'].to_numpy())
    return np.concatenate(oof), np.concatenate(val_true)

# Tree-based learners tuned via Optuna in Section 4.1. MLP/NaiveBayes were
# dropped (see Section 4 markdown) in favour of ExtraTrees, which is both
# literature-validated on this dataset and structurally decorrelated from the
# gradient-boosted trees (random rather than gradient-optimal splits).
TREE_AND_SIMPLE_LEARNERS = ['LightGBM', 'CatBoost', 'XGBoost', 'ExtraTrees']

oof_store = {}     # horizon -> {model: oof_prob}
test_store= {}     # horizon -> {model: test_prob}
for h in horizons:
    print(f'\n== OOF generation (horizon {h}) ==')
    s   = splits[h]
    folds = s['folds']
    oof_index = np.concatenate([fold['X_va'].index.to_numpy() for fold in folds])
    splits[h]['oof_index'] = oof_index
    X_tr, y_tr, X_te, y_te = s['X_tr'], s['y_tr'], s['X_te'], s['y_te']
    oof_store[h]  = {}; test_store[h] = {}
    for name in TREE_AND_SIMPLE_LEARNERS:
        params = hpo_results[h][name]['best_params']
        oof, oof_true = get_oof_for_learner(name, params, folds)
        p_te = predict_test_with_best_params(name, params, X_tr, y_tr, X_te)
        oof_store[h][name]  = oof
        test_store[h][name] = p_te
        print(f'   {name:11s}  OOF-AUC={roc_auc_score(oof_true, oof):.4f}  test-AUC={roc_auc_score(y_te, p_te):.4f}')
    # FT-Transformer
    if ftt_results[h]['best_cfg'] is not None:
        cfg = ftt_results[h]['best_cfg']
        # CV OOF for FT: for each fold, train on training rows, predict on validation rows
        oof_ft_pred = np.zeros(len(X_tr))
        oof_ft_true = np.zeros(len(X_tr), dtype=int)
        cursor = 0
        for fi, fold in enumerate(folds):
            model, sc, _ = train_ftt(fold['X_tr'], fold['y_tr'], fold['X_va'], fold['y_va'], **cfg)
            p_va = predict_ftt(model, sc, fold['X_va'])
            n = len(p_va)
            oof_ft_pred[cursor:cursor+n] = p_va
            oof_ft_true[cursor:cursor+n] = fold['y_va'].to_numpy()
            cursor += n
        # Train final FT-Transformer on full training data using a held-out subset as early-stop proxy
        # We split a tiny portion (10%) of training as a "val" so train_ftt can early-stop
        # without ever touching the test set (avoids leakage in early-stopping).
        from sklearn.model_selection import train_test_split
        Xfit_tr, Xfit_va, yfit_tr, yfit_va = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED)
        model_full, sc_full, _ = train_ftt(Xfit_tr, yfit_tr, Xfit_va, yfit_va, **cfg)
        p_te_ft = predict_ftt(model_full, sc_full, X_te)
        oof_store[h]['FTTransformer'] = oof_ft_pred
        test_store[h]['FTTransformer'] = p_te_ft
        print(f'   FT-Trans    OOF-AUC={roc_auc_score(oof_ft_true, oof_ft_pred):.4f}  test-AUC={roc_auc_score(y_te, p_te_ft):.4f}')
    # TabPFN (or proxy)
    if USE_TABPFN:
        oof_tp = np.zeros(len(X_tr))
        oof_tp_true = np.zeros(len(X_tr), dtype=int)
        cursor = 0
        for fi, fold in enumerate(folds):
            Xtr, ytr = fold['X_tr'].to_numpy(), fold['y_tr'].to_numpy()
            Xva, yva = fold['X_va'].to_numpy(), fold['y_va'].to_numpy()
            try:
                m = TabPFNClassifier(device=TABPFN_DEVICE, n_estimators=TABPFN_NE,
                                     random_state=SEED, ignore_pretraining_limits=True)
                m.fit(Xtr, ytr); p = m.predict_proba(Xva)[:,1]
            except Exception:
                m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=64, random_state=SEED, verbose=-1)
                m.fit(Xtr, ytr); p = m.predict_proba(Xva)[:,1]
            n = len(p); oof_tp[cursor:cursor+n] = p; oof_tp_true[cursor:cursor+n] = yva; cursor += n
        # Train final on full X_tr
        try:
            m_full = TabPFNClassifier(device=TABPFN_DEVICE, n_estimators=TABPFN_NE,
                                      random_state=SEED, ignore_pretraining_limits=True)
            m_full.fit(X_tr.to_numpy(), y_tr.to_numpy())
            p_te_tp = m_full.predict_proba(X_te.to_numpy())[:,1]
        except Exception:
            m_full = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=64, random_state=SEED, verbose=-1)
            m_full.fit(X_tr, y_tr); p_te_tp = m_full.predict_proba(X_te)[:,1]
        oof_store[h]['TabPFN'] = oof_tp
        test_store[h]['TabPFN'] = p_te_tp
        print(f'   TabPFN      OOF-AUC={roc_auc_score(oof_tp_true, oof_tp):.4f}  test-AUC={roc_auc_score(y_te, p_te_tp):.4f}')
print('\nOOF + test predictions stored for every base learner x horizon.')


Streaming output truncated to the last 5000 lines.
139:	learn: 0.2369716	total: 926ms	remaining: 1.01s
140:	learn: 0.2366461	total: 932ms	remaining: 1s
141:	learn: 0.2362672	total: 939ms	remaining: 998ms
142:	learn: 0.2359602	total: 945ms	remaining: 991ms
143:	learn: 0.2356500	total: 950ms	remaining: 983ms
144:	learn: 0.2352682	total: 958ms	remaining: 978ms
145:	learn: 0.2349604	total: 965ms	remaining: 972ms
146:	learn: 0.2345570	total: 972ms	remaining: 965ms
147:	learn: 0.2341707	total: 979ms	remaining: 959ms
148:	learn: 0.2338838	total: 985ms	remaining: 952ms
149:	learn: 0.2334827	total: 992ms	remaining: 946ms
150:	learn: 0.2333082	total: 999ms	remaining: 939ms
151:	learn: 0.2330731	total: 1.01s	remaining: 934ms
152:	learn: 0.2327114	total: 1.01s	remaining: 928ms
153:	learn: 0.2324549	total: 1.02s	remaining: 922ms
154:	learn: 0.2320076	total: 1.03s	remaining: 916ms
155:	learn: 0.2316816	total: 1.03s	remaining: 909ms
156:	learn: 0.2313540	total: 1.04s	remaining: 902ms
157:	learn: 0.23

In [ ]:
# ============================================================
# CELL 5.2  -  Save OOF matrices
# ============================================================
for h in horizons:
    cols = list(oof_store[h].keys())
    pd.DataFrame(oof_store[h])[cols].to_csv(f'{OUT_DIR}/csv/09_oof_probs_{h}.csv', index=False)
    pd.DataFrame(test_store[h])[cols].to_csv(f'{OUT_DIR}/csv/09_test_probs_{h}.csv', index=False)
print('OOF + test prob CSVs saved.')

OOF + test prob CSVs saved.


In [ ]:
# ============================================================
# CELL 5.3  -  Individual base-learner performance (Accuracy / Precision /
#              Recall / F1 / AUC) + comparison figures
#
# This directly addresses a gap in the original notebook: the stacking
# metrics (Section 6) reported only the *ensemble's* performance. Here we
# score every base learner on its own, on the same held-out test set, so we
# can see which base models are actually strong and whether the extra
# learners (MLP, NaiveBayes) are pulling their weight.
# ============================================================
import matplotlib.pyplot as plt

base_learner_rows = []
for h in horizons:
    y_te = splits[h]['y_te'].to_numpy()
    for name, p_te in test_store[h].items():
        y_pred = (p_te >= 0.5).astype(int)
        base_learner_rows.append({
            'horizon': h, 'model': name,
            'accuracy':  accuracy_score(y_te, y_pred),
            'precision': precision_score(y_te, y_pred, zero_division=0),
            'recall':    recall_score(y_te, y_pred, zero_division=0),
            'f1':        f1_score(y_te, y_pred, zero_division=0),
            'auc':       roc_auc_score(y_te, p_te),
        })
df_base_metrics = pd.DataFrame(base_learner_rows)
df_base_metrics.to_csv(f'{OUT_DIR}/csv/09b_base_learner_metrics.csv', index=False)
print(df_base_metrics.round(4).to_string(index=False))

models_all = list(dict.fromkeys(df_base_metrics['model'].tolist()))  # preserve first-seen order

# --- Figure 1: per-horizon Accuracy bar chart for every base learner ---
fig, axes = plt.subplots(1, 3, figsize=(19, 5), sharey=True)
palette = plt.cm.tab10(np.linspace(0, 1, len(models_all)))
for i, h in enumerate(horizons):
    sub = df_base_metrics[df_base_metrics['horizon'] == h].set_index('model').reindex(models_all)
    bars = axes[i].bar(sub.index, sub['accuracy'], color=palette)
    axes[i].set_title(f'Base-Learner Accuracy — {h} ({horizon_desc[h]})')
    axes[i].set_ylabel('Accuracy' if i == 0 else '')
    axes[i].set_ylim(0, 1.05)
    axes[i].tick_params(axis='x', rotation=40)
    axes[i].grid(alpha=0.3, axis='y')
    for bar, v in zip(bars, sub['accuracy']):
        if pd.notna(v):
            axes[i].text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
                         ha='center', fontsize=8)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/figures/12_base_learner_accuracy.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Saved 12_base_learner_accuracy.png')

# --- Figure 2: all metrics, averaged across horizons, grouped bars ---
metric_show = ['accuracy', 'precision', 'recall', 'f1', 'auc']
avg_metrics = df_base_metrics.groupby('model')[metric_show].mean().reindex(models_all)
fig2, ax2 = plt.subplots(1, 1, figsize=(13, 6))
x = np.arange(len(models_all)); width = 0.15
for k, m in enumerate(metric_show):
    ax2.bar(x + k*width, avg_metrics[m], width=width, label=m)
ax2.set_xticks(x + width*2)
ax2.set_xticklabels(models_all, rotation=25)
ax2.set_ylim(0, 1.05)
ax2.set_title('Base-Learner Performance (averaged across t0/t1/t2)')
ax2.legend(ncol=5, loc='lower center', bbox_to_anchor=(0.5, -0.25))
ax2.grid(alpha=0.3, axis='y')
fig2.tight_layout()
fig2.savefig(f'{OUT_DIR}/figures/13_base_learner_all_metrics.png', dpi=120, bbox_inches='tight')
plt.close(fig2)
print('Saved 13_base_learner_all_metrics.png')
print('\nAveraged base-learner metrics (all horizons):')
print(avg_metrics.round(4).sort_values('auc', ascending=False))


horizon         model  accuracy  precision  recall     f1    auc
     t0      LightGBM    0.7782     0.7412  0.6655 0.7013 0.8420
     t0      CatBoost    0.7824     0.7625  0.6444 0.6985 0.8445
     t0       XGBoost    0.7713     0.7323  0.6549 0.6914 0.8401
     t0    ExtraTrees    0.7631     0.7333  0.6197 0.6718 0.8278
     t0 FTTransformer    0.7507     0.7441  0.5528 0.6343 0.7987
     t0        TabPFN    0.7893     0.7549  0.6831 0.7172 0.8632
     t1      LightGBM    0.8967     0.8693  0.8662 0.8677 0.9485
     t1      CatBoost    0.8981     0.8750  0.8627 0.8688 0.9482
     t1       XGBoost    0.9050     0.8853  0.8697 0.8774 0.9487
     t1    ExtraTrees    0.9036     0.8821  0.8697 0.8759 0.9440
     t1 FTTransformer    0.8788     0.8403  0.8521 0.8462 0.9296
     t1        TabPFN    0.9063     0.8885  0.8697 0.8790 0.9542
     t2      LightGBM    0.9449     0.9420  0.9155 0.9286 0.9711
     t2      CatBoost    0.9339     0.9184  0.9120 0.9152 0.9718
     t2       XGBoost    

## Section 6 — Stacking Ensemble + Temperature Scaling (Meta-Learner: LightGBM vs CatBoost, best selected per horizon)

Two candidate meta-learners — **LightGBM** and **CatBoost** — are each tuned with Optuna over the same meta-feature matrix (base-learner probabilities + aggregate stats: mean/std/min/max/spread/entropy). The one with the higher cross-validated AUC is selected per horizon (Section 6.1) and used for the rest of the pipeline (temperature scaling, thresholding, etc.). Section 6.1b then checks whether the resulting stacking ensemble actually outperforms the best single base learner and a naive mean blend, using a base-learner correlation heatmap as a diversity diagnostic.

In [ ]:
# ============================================================
# CELL 6.1  -  Build meta features + Optuna HPO for TWO candidate
#              meta-learners (LightGBM and CatBoost) — the better one
#              (by CV-AUC) is selected per horizon.
# ============================================================
from scipy.optimize import minimize
import joblib


def add_meta_stats(P: np.ndarray):
    """Add mean, std, min, max, spread, entropy to the probability matrix."""
    mean = P.mean(axis=1, keepdims=True)
    std  = P.std(axis=1, keepdims=True)
    mn, mx = P.min(axis=1, keepdims=True), P.max(axis=1, keepdims=True)
    spread = mx - mn
    eps = 1e-9
    ent = -(P*np.log(P+eps) + (1-P)*np.log(1-P+eps)).mean(axis=1, keepdims=True)
    return np.hstack([P, mean, std, mn, mx, spread, ent]).astype(np.float32)

def youden_threshold(y_true, prob):
    fpr, tpr, thr = roc_curve(y_true, prob)
    j = tpr - fpr
    return float(thr[np.argmax(j)])

def temperature_scale(p, T):
    p = np.clip(p, 1e-6, 1-1e-6)
    logit = np.log(p/(1-p))
    pT = 1.0/(1.0 + np.exp(-logit/T))
    return pT

def _cv_auc_meta(model_fn, M, y, n_splits=N_SPLITS, seed=SEED):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    aucs = []
    for tr, va in cv.split(M, y):
        m = model_fn()
        m.fit(M[tr], y.iloc[tr])
        p = m.predict_proba(M[va])[:, 1]
        aucs.append(roc_auc_score(y.iloc[va], p))
    return float(np.mean(aucs))

stacking = {}
meta_comparison_rows = []
for h in horizons:
    print(f'\n== Stacking (horizon {h}) ==')
    X_tr, y_tr = splits[h]['X_tr'], splits[h]['y_tr']
    X_te, y_te = splits[h]['X_te'], splits[h]['y_te']
    P_tr_oof = np.column_stack([oof_store[h][m] for m in oof_store[h].keys()])
    P_te      = np.column_stack([test_store[h][m] for m in test_store[h].keys()])

    # Realign OOF rows: they're in fold-validation-concatenation order, not X_tr's
    # original row order, so they must be reordered to match y_tr before stacking.
    oof_index = splits[h]['oof_index']
    order = pd.Index(oof_index).get_indexer(X_tr.index)
    assert (order >= 0).all(), f'OOF index alignment failed for horizon {h}'
    P_tr_oof = P_tr_oof[order]

    M_tr = add_meta_stats(P_tr_oof)
    M_te = add_meta_stats(P_te)
    base_models = list(oof_store[h].keys())

    # ---------- Candidate 1: Optuna-tuned LightGBM meta-learner ----------
    def _lgbm_obj(trial):
        params = {
            'objective':'binary','verbose':-1,'random_state':SEED,
            'n_estimators': trial.suggest_int('n_estimators', 100, 800),
            'learning_rate':trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves':   trial.suggest_int('num_leaves', 8, 128),
            'max_depth':    trial.suggest_int('max_depth', 3, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda':trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree':trial.suggest_float('colsample_bytree', 0.6, 1.0),
        }
        return _cv_auc_meta(lambda: lgb.LGBMClassifier(**params), M_tr, y_tr)
    study_lgbm = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED),
                                     study_name=f'MetaLGBM_{h}')
    study_lgbm.optimize(_lgbm_obj, n_trials=OPTUNA_TRIALS_META, show_progress_bar=False)
    lgbm_meta_params = dict(study_lgbm.best_params)
    lgbm_meta_params.update({'objective': 'binary', 'verbose': -1, 'random_state': SEED})
    print(f'   [LightGBM meta] best CV-AUC = {study_lgbm.best_value:.4f}')

    # ---------- Candidate 2: Optuna-tuned CatBoost meta-learner ----------
    def _cat_obj(trial):
        params = {
            'iterations':   trial.suggest_int('iterations', 100, 800),
            'learning_rate':trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'depth':        trial.suggest_int('depth', 3, 10),
            'l2_leaf_reg':  trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
            'random_seed':  SEED, 'verbose': False,
        }
        return _cv_auc_meta(lambda: CatBoostClassifier(**params), M_tr, y_tr)
    study_cat = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED),
                                    study_name=f'MetaCatBoost_{h}')
    study_cat.optimize(_cat_obj, n_trials=OPTUNA_TRIALS_META, show_progress_bar=False)
    cat_meta_params = dict(study_cat.best_params)
    cat_meta_params.update({'random_seed': SEED, 'verbose': False})
    print(f'   [CatBoost meta] best CV-AUC = {study_cat.best_value:.4f}')

    # ---------- Select the winning meta-learner by CV-AUC ----------
    if study_lgbm.best_value >= study_cat.best_value:
        meta_type, meta_params, meta_cv_auc = 'LightGBM', lgbm_meta_params, study_lgbm.best_value
        meta_model = lgb.LGBMClassifier(**meta_params)
    else:
        meta_type, meta_params, meta_cv_auc = 'CatBoost', cat_meta_params, study_cat.best_value
        meta_model = CatBoostClassifier(**meta_params)
    print(f'   ===> Selected meta-learner: {meta_type} (CV-AUC={meta_cv_auc:.4f})')
    meta_comparison_rows.append({
        'horizon': h, 'lightgbm_meta_cv_auc': study_lgbm.best_value,
        'catboost_meta_cv_auc': study_cat.best_value, 'selected_meta_learner': meta_type,
    })

    meta_model.fit(M_tr, y_tr)
    p_train_meta = meta_model.predict_proba(M_tr)[:,1]   # OOF training fit
    p_test_meta  = meta_model.predict_proba(M_te)[:,1]

    # ----- temperature scaling on training meta probs -----
    def _temp_obj(T):
        pT = temperature_scale(p_train_meta, T[0])
        return -roc_auc_score(y_tr, pT)
    res = minimize(_temp_obj, x0=np.array([1.0]), bounds=[(0.05, 10.0)], method='L-BFGS-B')
    T_star = float(res.x[0]); print(f'   best temperature T* = {T_star:.4f}')
    p_train_cal = temperature_scale(p_train_meta, T_star)
    p_test_cal  = temperature_scale(p_test_meta, T_star)

    tau = youden_threshold(y_tr, p_train_cal)
    print(f'   Youden-J threshold tau* = {tau:.4f}')

    stacking[h] = {
        'base_models': base_models,
        'meta_model': meta_model, 'meta_type': meta_type, 'meta_params': meta_params, 'T': T_star,
        'p_train_meta': p_train_meta, 'p_train_cal': p_train_cal,
        'p_test_meta':  p_test_meta,  'p_test_cal':  p_test_cal,
        'tau': tau, 'M_tr': M_tr, 'M_te': M_te, 'P_tr_oof': P_tr_oof, 'P_te': P_te,
    }
    # Save meta-learner
    pd.DataFrame({
        'model': base_models + ['(aggregate)']*6,
        'meta_features': [f'p_{m.lower()}' for m in base_models] + ['mean','std','min','max','spread','entropy']
    }).to_csv(f'{OUT_DIR}/csv/10_meta_feature_list_{h}.csv', index=False)
    pd.DataFrame({'oof_prob': p_train_meta, 'cal_prob': p_train_cal, 'true': y_tr.to_numpy()})\
        .to_csv(f'{OUT_DIR}/csv/10_meta_oof_{h}.csv', index=False)
    pd.DataFrame({'test_prob': p_test_meta, 'cal_prob': p_test_cal, 'true': y_te.to_numpy()})\
        .to_csv(f'{OUT_DIR}/csv/10_meta_test_{h}.csv', index=False)
    joblib.dump({'meta_model': meta_model, 'meta_type': meta_type, 'T': T_star, 'tau': tau,
                 'base_models': base_models}, f'{OUT_DIR}/models/stacker_{h}.joblib')

df_meta_comparison = pd.DataFrame(meta_comparison_rows)
df_meta_comparison.to_csv(f'{OUT_DIR}/csv/10b_meta_learner_comparison.csv', index=False)
print('\nMeta-learner comparison (LightGBM vs CatBoost):')
print(df_meta_comparison.round(4).to_string(index=False))

# --- Figure: LightGBM-meta vs CatBoost-meta CV-AUC per horizon ---
fig_meta, ax_meta = plt.subplots(figsize=(8, 5))
x = np.arange(len(horizons)); width = 0.35
ax_meta.bar(x - width/2, df_meta_comparison['lightgbm_meta_cv_auc'], width, label='LightGBM meta-learner')
ax_meta.bar(x + width/2, df_meta_comparison['catboost_meta_cv_auc'], width, label='CatBoost meta-learner')
for i, row in df_meta_comparison.iterrows():
    winner_auc = max(row['lightgbm_meta_cv_auc'], row['catboost_meta_cv_auc'])
    ax_meta.annotate(row['selected_meta_learner'], (i, winner_auc + 0.002),
                     ha='center', fontsize=8, fontweight='bold')
ax_meta.set_xticks(x); ax_meta.set_xticklabels(horizons)
ax_meta.set_ylabel('Meta-learner CV-AUC')
ax_meta.set_title('Meta-Learner Comparison: LightGBM vs CatBoost (winner labeled)')
ax_meta.legend(); ax_meta.grid(alpha=0.3, axis='y')
fig_meta.tight_layout()
fig_meta.savefig(f'{OUT_DIR}/figures/14_meta_learner_comparison.png', dpi=120, bbox_inches='tight')
plt.close(fig_meta)
print('Saved 14_meta_learner_comparison.png, 10b_meta_learner_comparison.csv')
print('\nStacking done.')



== Stacking (horizon t0) ==
   [LightGBM meta] best CV-AUC = 0.8454
   [CatBoost meta] best CV-AUC = 0.8495
   ===> Selected meta-learner: CatBoost (CV-AUC=0.8495)
   best temperature T* = 1.0000
   Youden-J threshold tau* = 0.4037

== Stacking (horizon t1) ==
   [LightGBM meta] best CV-AUC = 0.9369
   [CatBoost meta] best CV-AUC = 0.9380
   ===> Selected meta-learner: CatBoost (CV-AUC=0.9380)
   best temperature T* = 1.0000
   Youden-J threshold tau* = 0.3672

== Stacking (horizon t2) ==
   [LightGBM meta] best CV-AUC = 0.9523
   [CatBoost meta] best CV-AUC = 0.9546
   ===> Selected meta-learner: CatBoost (CV-AUC=0.9546)
   best temperature T* = 1.0000
   Youden-J threshold tau* = 0.3422

Meta-learner comparison (LightGBM vs CatBoost):
horizon  lightgbm_meta_cv_auc  catboost_meta_cv_auc selected_meta_learner
     t0                0.8454                0.8495              CatBoost
     t1                0.9369                0.9380              CatBoost
     t2                0.9523 

### 6.1c  TARB — TabPFN-Anchored Residual Boosting (alternative backbone)

The LightGBM/CatBoost meta-stacker above (Section 6.1) rarely beats the single best base
learner by much — free blend weights can always slightly overfit the OOF matrix without
actually adding information. **TARB** tries a structurally different idea instead of a
better blend weight:

- A LightGBM booster's raw margin (`init_score`) is **initialised at TabPFN's own
  logit-space prediction** for every row.
- A **small, heavily-regularised** set of trees (≤150 trees, depth ≤4, strong
  `reg_alpha`/`reg_lambda`) is then fit to the *residual* — i.e. only what TabPFN gets wrong.
- Where TabPFN is already near-optimal, the gradient the booster sees is ~0, so it adds
  (almost) nothing there — unlike a free blend weight, TARB has no parameter that can make
  things *worse* than TabPFN alone; at worst it collapses back to TabPFN's own predictions.

**No-leakage guarantees (all four hold simultaneously):**
1. The TabPFN anchor for every training row is its **out-of-fold** prediction (the exact
   `P_tr_oof` column already produced for the stacking meta-learner) — never a TabPFN fit
   that saw that row.
2. Optuna HPO and calibration for the residual booster use a **nested walk over the same 5
   outer folds** used everywhere else, refit from scratch each time, on each fold's
   **un-resampled** training partition (SMOTE rows have no genuine TabPFN anchor, so they're
   excluded from TARB's own fitting to avoid reintroducing synthetic-row leakage/noise).
3. The held-out test set `X_te`/`y_te` is touched **exactly once**, at the very end, after
   HPO + calibration are already frozen — and the anchor added there is TabPFN's prediction
   from a TabPFN model that was itself fit only on `X_tr`.
4. TARB only **replaces** the incumbent stacking backbone for a horizon if its nested-CV AUC
   actually beats the already-selected LightGBM-meta/CatBoost-meta CV-AUC — so downstream
   sections (conformal prediction, rejection, multi-task heads, at-risk tables) pick it up
   automatically when it wins, and fall back to the existing stacker when it doesn't.


In [ ]:
# ============================================================
# CELL 6.1c  -  TabPFN-Anchored Residual Boosting (TARB)
#   Alternative ensembling backbone to the LightGBM/CatBoost
#   stacking meta-learner tuned in Cell 6.1.
#
#   init_score(x) = logit(TabPFN(x))   <-  booster's raw margin starts here
#   F(x)          = init_score(x) + sum_of_a_few_shallow_trees(x)
#   p(x)          = sigmoid(F(x))
#
#   See the markdown cell above for the full leakage-control rationale.
#   Summary: every anchor value used to FIT the residual booster (both
#   during nested-CV HPO and in the final full-training-set fit) is an
#   out-of-fold TabPFN prediction; X_te is scored exactly once, at the
#   very end, with an anchor from a TabPFN model that only ever saw X_tr.
# ============================================================
from scipy.optimize import minimize as _tarb_minimize   # safe re-import, already imported in 6.1
import lightgbm as lgb


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def tarb_search_space(trial):
    # Deliberately narrow / shrinkage-biased: this booster should only ever
    # nibble at TabPFN's residual, never build a competing full model.
    return {
        'objective':        'binary',
        'verbose':          -1,
        'random_state':     SEED,
        'n_estimators':     trial.suggest_int('n_estimators', 20, 150),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.10, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 3, 15),
        'max_depth':        trial.suggest_int('max_depth', 2, 4),
        'min_child_samples':trial.suggest_int('min_child_samples', 20, 100),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq':   1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }


def fit_tarb_booster(X, y, anchor_logit, params):
    p = dict(params)
    n_estimators = p.pop('n_estimators')
    dtrain = lgb.Dataset(X, label=np.asarray(y), init_score=np.asarray(anchor_logit),
                          free_raw_data=False)
    return lgb.train(p, dtrain, num_boost_round=n_estimators)


def tarb_predict_proba(booster, X, anchor_logit):
    raw = booster.predict(X, raw_score=True)
    return _sigmoid(raw + np.asarray(anchor_logit))


tarb_results = {}
for h in horizons:
    base_models_h = stacking[h]['base_models']
    if 'TabPFN' not in base_models_h:
        print(f'[TARB] {h}: no TabPFN column available -> skipping TARB for this horizon.')
        continue

    print(f'\n== TARB (horizon {h}) ==')
    X_tr, y_tr = splits[h]['X_tr'], splits[h]['y_tr']
    X_te, y_te = splits[h]['X_te'], splits[h]['y_te']
    folds = splits[h]['folds']

    # TabPFN's OOF column from the exact same P_tr_oof matrix the meta-learner
    # used (already row-aligned to X_tr.index by Cell 6.1) -> logit space.
    tp_idx = base_models_h.index('TabPFN')
    anchor_train_prob = stacking[h]['P_tr_oof'][:, tp_idx]
    anchor_test_prob  = stacking[h]['P_te'][:, tp_idx]           # row-aligned to X_te already
    anchor_train_series = pd.Series(_logit(anchor_train_prob), index=X_tr.index)
    anchor_test_logit   = _logit(anchor_test_prob)

    # ---- rebuild each outer fold's UN-RESAMPLED train partition ----
    # folds[i]['X_tr'] went through Borderline-SMOTE + Tomek and had its index
    # reset (Cell 3.1), so synthetic rows can't be matched to a genuine TabPFN
    # OOF anchor. TARB instead trains on the real rows only, reconstructed
    # directly from the untouched X_tr / fold['X_va'].index.
    unsampled_folds = []
    for fold in folds:
        va_idx = fold['X_va'].index
        tr_idx = X_tr.index.difference(va_idx)
        unsampled_folds.append({
            'X_tr': X_tr.loc[tr_idx], 'y_tr': y_tr.loc[tr_idx],
            'X_va': fold['X_va'],     'y_va': fold['y_va'],
        })

    def _cv_auc_tarb(params):
        aucs = []
        for fold in unsampled_folds:
            a_tr = anchor_train_series.loc[fold['X_tr'].index].to_numpy()
            a_va = anchor_train_series.loc[fold['X_va'].index].to_numpy()
            booster = fit_tarb_booster(fold['X_tr'], fold['y_tr'], a_tr, params)
            p_va = tarb_predict_proba(booster, fold['X_va'], a_va)
            aucs.append(roc_auc_score(fold['y_va'], p_va))
        return float(np.mean(aucs))

    def _tarb_obj(trial):
        return _cv_auc_tarb(tarb_search_space(trial))

    study_tarb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED),
                                      study_name=f'TARB_{h}')
    study_tarb.optimize(_tarb_obj, n_trials=OPTUNA_TRIALS_META, show_progress_bar=False)
    tarb_params = dict(study_tarb.best_params)
    tarb_params.update({'objective': 'binary', 'verbose': -1, 'random_state': SEED})
    tarb_cv_auc = study_tarb.best_value
    print(f'   [TARB] best nested CV-AUC = {tarb_cv_auc:.4f}  params={study_tarb.best_params}')

    # ---- honest OOF predictions at the winning hyperparameters, used ONLY
    #      for temperature scaling / threshold calibration (X_te not involved) ----
    oof_tarb = pd.Series(index=X_tr.index, dtype=float)
    for fold in unsampled_folds:
        a_tr = anchor_train_series.loc[fold['X_tr'].index].to_numpy()
        a_va = anchor_train_series.loc[fold['X_va'].index].to_numpy()
        booster = fit_tarb_booster(fold['X_tr'], fold['y_tr'], a_tr, tarb_params)
        p_va = tarb_predict_proba(booster, fold['X_va'], a_va)
        oof_tarb.loc[fold['X_va'].index] = p_va
    oof_tarb = oof_tarb.loc[X_tr.index].to_numpy()
    print(f'   [TARB] OOF-AUC (calibration target) = {roc_auc_score(y_tr, oof_tarb):.4f}')

    # ---- final booster: ONE fit on the full training set, anchored at
    #      TabPFN's OOF logit (never TabPFN's in-sample fit) ----
    anchor_full = anchor_train_series.loc[X_tr.index].to_numpy()
    final_booster = fit_tarb_booster(X_tr, y_tr, anchor_full, tarb_params)
    p_test_tarb = tarb_predict_proba(final_booster, X_te, anchor_test_logit)  # X_te touched ONLY here

    # ---- temperature scaling + Youden threshold, fit on the OOF probs only ----
    def _temp_obj_tarb(T):
        pT = temperature_scale(oof_tarb, T[0])
        return -roc_auc_score(y_tr, pT)
    res_t = _tarb_minimize(_temp_obj_tarb, x0=np.array([1.0]), bounds=[(0.05, 10.0)], method='L-BFGS-B')
    T_tarb = float(res_t.x[0])
    p_train_cal_tarb = temperature_scale(oof_tarb, T_tarb)
    p_test_cal_tarb  = temperature_scale(p_test_tarb, T_tarb)
    tau_tarb = youden_threshold(y_tr, p_train_cal_tarb)
    test_auc_tarb = roc_auc_score(y_te, p_test_cal_tarb)
    print(f'   [TARB] T*={T_tarb:.4f}  tau*={tau_tarb:.4f}  TEST-AUC={test_auc_tarb:.4f}')

    tarb_results[h] = {
        'cv_auc': tarb_cv_auc, 'test_auc': test_auc_tarb,
        'params': tarb_params, 'T': T_tarb, 'tau': tau_tarb,
        'p_train_cal': p_train_cal_tarb, 'p_test_cal': p_test_cal_tarb,
        'booster': final_booster,
    }

    # ---- only adopt TARB as the horizon's backbone if it beats the
    #      already-selected LightGBM-meta / CatBoost-meta on CV-AUC ----
    incumbent_row = next(r for r in meta_comparison_rows if r['horizon'] == h)
    incumbent_cv_auc = max(incumbent_row['lightgbm_meta_cv_auc'], incumbent_row['catboost_meta_cv_auc'])

    if tarb_cv_auc > incumbent_cv_auc:
        print(f'   ===> TARB WINS for {h} (CV-AUC {tarb_cv_auc:.4f} > {incumbent_cv_auc:.4f} incumbent) -> adopting as backbone.')
        stacking[h].update({
            'meta_type':   'TARB',
            'meta_params': tarb_params,
            'meta_model':  final_booster,
            'T':           T_tarb,
            'p_train_meta': oof_tarb,        'p_train_cal': p_train_cal_tarb,
            'p_test_meta':  p_test_tarb,      'p_test_cal':  p_test_cal_tarb,
            'tau': tau_tarb,
        })
        joblib.dump({'booster': final_booster, 'meta_type': 'TARB', 'T': T_tarb, 'tau': tau_tarb,
                     'base_models': base_models_h, 'tabpfn_col': tp_idx},
                    f'{OUT_DIR}/models/stacker_{h}.joblib')
    else:
        print(f'   ===> TARB does not beat incumbent for {h} '
              f'(CV-AUC {tarb_cv_auc:.4f} <= {incumbent_cv_auc:.4f}) -> keeping {incumbent_row["selected_meta_learner"]}-meta.')

# ---- comparison table + figure ----
tarb_summary_rows = []
for h in horizons:
    incumbent_row = next(r for r in meta_comparison_rows if r['horizon'] == h)
    incumbent_cv_auc = max(incumbent_row['lightgbm_meta_cv_auc'], incumbent_row['catboost_meta_cv_auc'])
    row = {
        'horizon': h,
        'incumbent_meta_learner': incumbent_row['selected_meta_learner'],
        'incumbent_cv_auc': incumbent_cv_auc,
        'tarb_cv_auc': tarb_results[h]['cv_auc'] if h in tarb_results else np.nan,
        'tarb_test_auc': tarb_results[h]['test_auc'] if h in tarb_results else np.nan,
        'backbone_selected': stacking[h]['meta_type'],
    }
    tarb_summary_rows.append(row)
df_tarb_summary = pd.DataFrame(tarb_summary_rows)
df_tarb_summary.to_csv(f'{OUT_DIR}/csv/10c_tarb_vs_incumbent.csv', index=False)
print('\nTARB vs incumbent meta-learner comparison:')
print(df_tarb_summary.round(4).to_string(index=False))

fig_tarb, ax_tarb = plt.subplots(figsize=(8, 5))
xw = np.arange(len(horizons)); width = 0.35
ax_tarb.bar(xw - width/2, df_tarb_summary['incumbent_cv_auc'], width, label='Incumbent meta-learner (CV-AUC)')
ax_tarb.bar(xw + width/2, df_tarb_summary['tarb_cv_auc'], width, label='TARB (nested CV-AUC)')
for i, row in df_tarb_summary.iterrows():
    ax_tarb.annotate(row['backbone_selected'], (i, max(row['incumbent_cv_auc'], row['tarb_cv_auc'] or 0) + 0.002),
                      ha='center', fontsize=8, fontweight='bold')
ax_tarb.set_xticks(xw); ax_tarb.set_xticklabels(horizons)
ax_tarb.set_ylabel('CV-AUC'); ax_tarb.set_title('TARB vs Incumbent Stacking Meta-Learner (winner labeled)')
ax_tarb.legend(); ax_tarb.grid(alpha=0.3, axis='y')
fig_tarb.tight_layout()
fig_tarb.savefig(f'{OUT_DIR}/figures/16b_tarb_vs_incumbent.png', dpi=120, bbox_inches='tight')
plt.close(fig_tarb)
print('Saved 16b_tarb_vs_incumbent.png, 10c_tarb_vs_incumbent.csv')
print('\nTARB done. `stacking[h]` now reflects whichever backbone (meta-stacker or TARB) won per horizon,')
print('so every downstream section (7-14) automatically uses the winner.')



== TARB (horizon t0) ==
   [TARB] best nested CV-AUC = 0.8460  params={'n_estimators': 22, 'learning_rate': 0.012626677949581612, 'num_leaves': 13, 'max_depth': 4, 'min_child_samples': 20, 'reg_alpha': 9.841577263824066, 'reg_lambda': 0.10644003960788526, 'subsample': 0.6145219252928473, 'colsample_bytree': 0.7731293015521309}
   [TARB] OOF-AUC (calibration target) = 0.8448
   [TARB] T*=1.0000  tau*=0.4352  TEST-AUC=0.8627
   ===> TARB does not beat incumbent for t0 (CV-AUC 0.8460 <= 0.8495) -> keeping CatBoost-meta.

== TARB (horizon t1) ==
   [TARB] best nested CV-AUC = 0.9355  params={'n_estimators': 22, 'learning_rate': 0.012626677949581612, 'num_leaves': 13, 'max_depth': 4, 'min_child_samples': 20, 'reg_alpha': 9.841577263824066, 'reg_lambda': 0.10644003960788526, 'subsample': 0.6145219252928473, 'colsample_bytree': 0.7731293015521309}
   [TARB] OOF-AUC (calibration target) = 0.9340
   [TARB] T*=1.0000  tau*=0.4930  TEST-AUC=0.9543
   ===> TARB does not beat incumbent for t1 (CV-

In [ ]:
# ============================================================
# CELL 6.1b - Base-learner diversity check + "does stacking actually help?"
#             validation. This directly checks whether the stacking
#             combination is worthwhile versus (a) the single best base
#             learner and (b) a naive unweighted average of all base learners.
# ============================================================
import seaborn as sns

corr_fig, corr_axes = plt.subplots(1, 3, figsize=(19, 5))
stack_value_rows = []
for i, h in enumerate(horizons):
    base_models = stacking[h]['base_models']
    P_te = stacking[h]['P_te']
    y_te = splits[h]['y_te'].to_numpy()

    df_corr = pd.DataFrame(P_te, columns=base_models).corr()
    sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
                ax=corr_axes[i], cbar=(i == 2))
    corr_axes[i].set_title(f'Base-Learner Prediction Correlation — {h}')

    single_aucs = {m: roc_auc_score(y_te, P_te[:, j]) for j, m in enumerate(base_models)}
    best_single_name = max(single_aucs, key=single_aucs.get)
    best_single_auc = single_aucs[best_single_name]
    mean_blend_auc = roc_auc_score(y_te, P_te.mean(axis=1))
    stack_auc = roc_auc_score(y_te, stacking[h]['p_test_cal'])
    stack_value_rows.append({
        'horizon': h,
        'best_single_model': best_single_name, 'best_single_auc': best_single_auc,
        'mean_blend_auc': mean_blend_auc,
        'stacking_auc': stack_auc, 'meta_learner': stacking[h]['meta_type'],
        'gain_vs_best_single': stack_auc - best_single_auc,
        'gain_vs_mean_blend': stack_auc - mean_blend_auc,
    })
corr_fig.tight_layout()
corr_fig.savefig(f'{OUT_DIR}/figures/15_base_learner_correlation.png', dpi=120, bbox_inches='tight')
plt.close(corr_fig)

df_stack_value = pd.DataFrame(stack_value_rows)
df_stack_value.to_csv(f'{OUT_DIR}/csv/12_stacking_vs_baselines.csv', index=False)
print(df_stack_value.round(4).to_string(index=False))

fig_val, ax_val = plt.subplots(figsize=(9, 5))
x = np.arange(len(horizons)); width = 0.25
ax_val.bar(x - width, df_stack_value['best_single_auc'], width, label='Best single base learner')
ax_val.bar(x,          df_stack_value['mean_blend_auc'], width, label='Simple mean blend')
ax_val.bar(x + width,   df_stack_value['stacking_auc'],   width, label='Stacking ensemble')
ax_val.set_xticks(x); ax_val.set_xticklabels(horizons)
ax_val.set_ylabel('Test AUC')
ax_val.set_title('Is the stacking combination worth it?')
ax_val.legend(); ax_val.grid(alpha=0.3, axis='y')
fig_val.tight_layout()
fig_val.savefig(f'{OUT_DIR}/figures/16_stacking_vs_baselines.png', dpi=120, bbox_inches='tight')
plt.close(fig_val)
print('Saved 15_base_learner_correlation.png, 16_stacking_vs_baselines.png, 12_stacking_vs_baselines.csv')

for _, row in df_stack_value.iterrows():
    verdict = 'YES' if row['gain_vs_best_single'] > 0 and row['gain_vs_mean_blend'] > 0 else 'MARGINAL / NO'
    print(f"[{row['horizon']}] stacking ({row['meta_learner']} meta) beats best single learner by "
          f"{row['gain_vs_best_single']:+.4f} AUC and mean blend by {row['gain_vs_mean_blend']:+.4f} AUC "
          f"-> worthwhile: {verdict}")


horizon best_single_model  best_single_auc  mean_blend_auc  stacking_auc meta_learner  gain_vs_best_single  gain_vs_mean_blend
     t0            TabPFN           0.8632          0.8537        0.8545     CatBoost              -0.0087              0.0008
     t1            TabPFN           0.9542          0.9493        0.9469     CatBoost              -0.0073             -0.0024
     t2            TabPFN           0.9774          0.9741        0.9736     CatBoost              -0.0037             -0.0005
Saved 15_base_learner_correlation.png, 16_stacking_vs_baselines.png, 12_stacking_vs_baselines.csv
[t0] stacking (CatBoost meta) beats best single learner by -0.0087 AUC and mean blend by +0.0008 AUC -> worthwhile: MARGINAL / NO
[t1] stacking (CatBoost meta) beats best single learner by -0.0073 AUC and mean blend by -0.0024 AUC -> worthwhile: MARGINAL / NO
[t2] stacking (CatBoost meta) beats best single learner by -0.0037 AUC and mean blend by -0.0005 AUC -> worthwhile: MARGINAL / NO


In [ ]:
# ============================================================
# CELL 6.2  -  Stacking metrics: Acc/Prec/Rec/F1/AUC + Brier, LogLoss, Confusion
# ============================================================
import matplotlib.pyplot as plt, seaborn as sns

metrics_rows = []
fig_cm, axes_cm = plt.subplots(1, 3, figsize=(15, 4))
for i, h in enumerate(horizons):
    s = stacking[h]
    p_test = s['p_test_cal']; y_te = splits[h]['y_te'].to_numpy()
    y_pred = (p_test >= s['tau']).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    auc  = roc_auc_score(y_te, p_test)
    brier= brier_score_loss(y_te, np.clip(p_test,1e-6,1-1e-6))
    ll   = log_loss(y_te, np.clip(p_test,1e-6,1-1e-6))
    metrics_rows.append({
        'horizon':h, 'description':horizon_desc[h],
        'tau':s['tau'], 'T':s['T'],
        'accuracy':acc, 'precision':prec, 'recall':rec,
        'f1':f1, 'auc':auc, 'brier':brier, 'logloss':ll,
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
    })
    sns.heatmap(np.array([[tn,fp],[fn,tp]]), annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'],
                ax=axes_cm[i])
    axes_cm[i].set_title(f'Confusion {h}')
fig_cm.tight_layout()
fig_cm.savefig(f'{OUT_DIR}/figures/01_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.close(fig_cm)
df_metrics = pd.DataFrame(metrics_rows)
df_metrics.to_csv(f'{OUT_DIR}/csv/11_stacking_metrics.csv', index=False)
print(df_metrics.round(4))
print('Saved 01_confusion_matrices.png and 11_stacking_metrics.csv')

  horizon   description     tau    T  accuracy  precision  recall      f1  \
0      t0  pre_semester  0.4037  1.0    0.7851     0.7388  0.6972  0.7174   
1      t1    after_sem1  0.3672  1.0    0.8871     0.8389  0.8803  0.8591   
2      t2    after_sem2  0.3422  1.0    0.9160     0.8562  0.9437  0.8978   

      auc   brier  logloss   TN  FP  FN   TP  
0  0.8545  0.1461   0.4507  372  70  86  198  
1  0.9469  0.0755   0.2591  394  48  34  250  
2  0.9736  0.0581   0.2368  397  45  16  268  
Saved 01_confusion_matrices.png and 11_stacking_metrics.csv


In [ ]:
# ============================================================
# CELL 6.3  -  ROC overlays across horizons + per-horizon ROCs
# ============================================================
fig_roc, ax_roc = plt.subplots(1, 1, figsize=(8,6))
for h in horizons:
    p_test = stacking[h]['p_test_cal']; y_te = splits[h]['y_te'].to_numpy()
    fpr, tpr, _ = roc_curve(y_te, p_test)
    ax_roc.plot(fpr, tpr, label=f'{h} ({horizon_desc[h]}) AUC={roc_auc_score(y_te,p_test):.3f}', linewidth=2)
ax_roc.plot([0,1],[0,1], 'k--', alpha=0.4)
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC — Stacking Ensemble across t0/t1/t2')
ax_roc.legend(loc='lower right'); ax_roc.grid(alpha=0.3)
fig_roc.tight_layout()
fig_roc.savefig(f'{OUT_DIR}/figures/02_roc_overlay_horizons.png', dpi=120, bbox_inches='tight')
plt.close(fig_roc)
print('Saved 02_roc_overlay_horizons.png')

# Per-horizon calibration reliability diagrams
fig_cal, axes_cal = plt.subplots(1, 3, figsize=(15,4))
for i,h in enumerate(horizons):
    p_test = stacking[h]['p_test_cal']; y_te = splits[h]['y_te'].to_numpy()
    bins = np.linspace(0,1,11)
    digitized = np.digitize(p_test, bins)-1
    bin_centres = []; bin_true = []
    for b in range(10):
        m = (digitized==b)
        if m.sum() > 0:
            bin_centres.append(p_test[m].mean()); bin_true.append(y_te[m].mean())
    axes_cal[i].plot([0,1],[0,1], 'k--')
    axes_cal[i].scatter(bin_centres, bin_true, color='C0')
    axes_cal[i].set_title(f'Reliability {h}')
    axes_cal[i].set_xlabel('Predicted'); axes_cal[i].set_ylabel('Observed')
    axes_cal[i].grid(alpha=0.3)
fig_cal.tight_layout()
fig_cal.savefig(f'{OUT_DIR}/figures/03_reliability_diagrams.png', dpi=120, bbox_inches='tight')
plt.close(fig_cal)
print('Saved 03_reliability_diagrams.png')

Saved 02_roc_overlay_horizons.png
Saved 03_reliability_diagrams.png


## Section 7 — Conformal Prediction (Mondrian-style + per-horizon)

In [ ]:
# ============================================================
# CELL 7.1  -  Conformal Prediction (per horizon)
# ============================================================
conformal_summary = []
for h in horizons:
    p_test = stacking[h]['p_test_cal']; y_te = splits[h]['y_te'].to_numpy()
    # Use half of the training OOF probabilities as calibration set, the other half as fitting
    p_train_calprob = stacking[h]['p_train_cal']; y_train = splits[h]['y_tr'].to_numpy()
    n = len(p_train_calprob); cal_n = n//2
    perm = np.random.RandomState(SEED).permutation(n)
    cal_scores = 1.0 - p_train_calprob[perm[:cal_n]][y_train[perm[:cal_n]] == 1]
    cal_scores = np.concatenate([cal_scores, p_train_calprob[perm[:cal_n]][y_train[perm[:cal_n]] == 0]])
    alpha = 0.10
    qhat = np.quantile(cal_scores, 1 - alpha, method='higher')
    # prediction sets: include class 1 if (1 - p) <= qhat, include class 0 if p <= qhat
    one_in = (1 - p_test) <= qhat
    zero_in = p_test <= qhat
    pred_sets = np.stack([zero_in, one_in], axis=1).astype(int)
    # marginal coverage
    cover = np.mean([y_te[i] in [j for j,v in enumerate(pred_sets[i]) if v==1] for i in range(len(y_te))])
    # average set size
    avg_size = float(np.mean(pred_sets.sum(axis=1)))
    conformal_summary.append({'horizon':h,'alpha':alpha,'qhat':float(qhat),
                              'coverage':float(cover),'avg_set_size':avg_size})
    np.save(f'{OUT_DIR}/meta/conformal_preds_{h}.npy', pred_sets)
pd.DataFrame(conformal_summary).to_csv(f'{OUT_DIR}/csv/12_conformal_summary.csv', index=False)
print(pd.DataFrame(conformal_summary).round(4))
print('Saved 12_conformal_summary.csv')

  horizon  alpha    qhat  coverage  avg_set_size
0      t0    0.1  0.6774    0.9008        1.2741
1      t1    0.1  0.4525    0.8953        0.9862
2      t2    0.1  0.4053    0.9187        0.9628
Saved 12_conformal_summary.csv


## Section 8 — Open-World Rejection (Out-of-Distribution Detection)

In [ ]:
# ============================================================
# CELL 8.1  -  Open-World Rejection via base-model disagreement
# ============================================================
from sklearn.mixture import GaussianMixture
reject_summary = []
fig_rej, axes_rej = plt.subplots(1, 3, figsize=(15,4))
for i, h in enumerate(horizons):
    P_te = stacking[h]['P_te']   # (n_test, n_base_models)
    # Uncertainty proxy: 1) entropy of base predictions; 2) std across base models
    eps = 1e-9
    ent = -(P_te*np.log(P_te+eps) + (1-P_te)*np.log(1-P_te+eps)).mean(axis=1)
    std = P_te.std(axis=1)
    sp  = P_te.max(axis=1) - P_te.min(axis=1)
    # Fit 2-component GMM on std (low vs high disagreement)
    feat = std.reshape(-1,1)
    gmm  = GaussianMixture(n_components=2, random_state=SEED).fit(feat)
    post = gmm.predict_proba(feat)
    # The component with higher mean belongs to high-uncertainty (OOD-like) cases
    means = gmm.means_.flatten(); high = int(np.argmax(means))
    ood_score = post[:, high]
    # Choose threshold at 90th percentile
    tau_rej = float(np.quantile(ood_score, 0.90))
    is_reject = ood_score >= tau_rej
    reject_summary.append({'horizon':h,'rejection_threshold':tau_rej,
                           'reject_rate':float(is_reject.mean()),
                           'ood_mean':float(ood_score.mean())})
    pd.DataFrame({'entropy':ent, 'std':std, 'spread':sp, 'ood_score':ood_score,
                  'reject_flag':is_reject.astype(int)})\
        .to_csv(f'{OUT_DIR}/csv/13_ood_scores_{h}.csv', index=False)
    axes_rej[i].hist(ood_score, bins=30, color='C2', alpha=0.7)
    axes_rej[i].axvline(tau_rej, color='red', linestyle='--')
    axes_rej[i].set_title(f'OOD score — {h}'); axes_rej[i].grid(alpha=0.3)
fig_rej.tight_layout()
fig_rej.savefig(f'{OUT_DIR}/figures/04_open_world_rejection.png', dpi=120, bbox_inches='tight')
plt.close(fig_rej)
pd.DataFrame(reject_summary).round(4).to_csv(f'{OUT_DIR}/csv/13_open_world_summary.csv', index=False)
print(pd.DataFrame(reject_summary).round(4))
print('Saved 04_open_world_rejection.png, 13_open_world_summary.csv, 13_ood_scores_*.csv')

  horizon  rejection_threshold  reject_rate  ood_mean
0      t0               0.9935       0.1006    0.2819
1      t1               0.9999       0.1006    0.2586
2      t2               1.0000       0.1088    0.3331
Saved 04_open_world_rejection.png, 13_open_world_summary.csv, 13_ood_scores_*.csv


## Section 9 — SHAP Explainability + Counterfactuals (per horizon)

In [ ]:
# ============================================================
# CELL 9.1  -  SHAP global summary, dependence, bar plot (per horizon)
# ============================================================
import shap
shap_results = {}
for h in horizons:
    print(f'\n== SHAP for horizon {h} ==')
    # Refit best LightGBM base learner on whole training set for interpretability
    params = hpo_results[h]['LightGBM']['best_params']
    # objective / random_state / verbose are already inside params
    m = lgb.LGBMClassifier(**params)
    m.fit(splits[h]['X_tr'], splits[h]['y_tr'])
    explainer = shap.TreeExplainer(m)
    sv = explainer.shap_values(splits[h]['X_te'])
    # In SHAP >=0.45 the result for binary is a list [class0, class1]
    if isinstance(sv, list):
        sv = sv[1]
    # global mean(|SHAP|) and mean(SHAP)
    mean_abs = np.abs(sv).mean(axis=0)
    mean_val = sv.mean(axis=0)
    cols = splits[h]['X_tr'].columns
    df_imp = pd.DataFrame({
        'feature': cols, 'mean_abs_shap': mean_abs, 'mean_shap': mean_val
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    df_imp.to_csv(f'{OUT_DIR}/csv/14_shap_global_importance_{h}.csv', index=False)
    shap_results[h] = {'explainer': explainer, 'shap_values': sv, 'columns': list(cols), 'model': m}
    # Summary plot
    plt.figure(figsize=(10,6))
    shap.summary_plot(sv, splits[h]['X_te'], feature_names=cols, show=False, max_display=20)
    plt.title(f'SHAP Summary — {h} ({horizon_desc[h]})')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/figures/05_shap_summary_{h}.png', dpi=120, bbox_inches='tight')
    plt.close()
    # Bar plot
    plt.figure(figsize=(9,5))
    top = df_imp.head(15)
    plt.barh(top['feature'][::-1], top['mean_abs_shap'][::-1], color='C3')
    plt.title(f'Top-15 SHAP — {h} ({horizon_desc[h]})')
    plt.xlabel('mean |SHAP|')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/figures/06_shap_bar_{h}.png', dpi=120, bbox_inches='tight')
    plt.close()
    # Dependence plots for top-3
    for j, feat in enumerate(df_imp['feature'].head(3).tolist()):
        plt.figure(figsize=(6,4))
        shap.dependence_plot(feat, sv, splits[h]['X_te'], feature_names=cols, show=False)
        plt.title(f'Dependence — {feat} ({h})')
        plt.tight_layout()
        plt.savefig(f'{OUT_DIR}/figures/07_shap_dependence_{h}_{j}_{feat.replace("/","_")}.png', dpi=120, bbox_inches='tight')
        plt.close()
print('SHAP artefacts saved for every horizon.')


== SHAP for horizon t0 ==

== SHAP for horizon t1 ==

== SHAP for horizon t2 ==
SHAP artefacts saved for every horizon.


<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

In [ ]:
# ============================================================
# CELL 9.2  -  Counterfactual (perturbation) explanations
# ============================================================
def counterfactual_flip(model, X_row, feature_names, base_p, delta=0.5, max_features=8):
    """Very simple, gradient-free counterfactual: nudge each feature by ±delta and
    return the smallest set that pushes the dropout probability below 0.5."""
    x = X_row.values.flatten().copy()
    p_curr = base_p
    if p_curr < 0.5:
        return None
    rng = np.random.RandomState(SEED)
    candidate_features = rng.choice(len(x), size=min(max_features, len(x)), replace=False)
    best_change = None
    for f in candidate_features:
        for sign in (+1,-1):
            x_try = x.copy()
            x_try[f] += sign * abs(x[f]) * delta + 1e-3*np.sign(sign)
            p_new = model.predict_proba(x_try.reshape(1,-1))[:,1][0]
            if p_new < p_curr:
                if best_change is None or p_new < best_change['p_new']:
                    best_change = {'feature_idx':int(f),
                                   'feature':feature_names[f],
                                   'new_val':float(x_try[f]),
                                   'old_val':float(x[f]),
                                   'p_new':float(p_new)}
    return best_change

cf_records = []
for h in horizons:
    print(f'\n== Counterfactual explanations (horizon {h}) ==')
    params = hpo_results[h]['LightGBM']['best_params']
    # objective / random_state / verbose are already inside params
    m = lgb.LGBMClassifier(**params)
    m.fit(splits[h]['X_tr'], splits[h]['y_tr'])
    Xte = splits[h]['X_te']
    cols = list(Xte.columns)
    # pick top-20 high-risk test students (highest predicted prob)
    probs = stacking[h]['p_test_cal']
    idx_sorted = np.argsort(-probs)[:25]
    for idx in idx_sorted:
        x = Xte.iloc[idx]
        cf = counterfactual_flip(m, x, cols, probs[idx])
        row = {'horizon':h, 'test_index':int(idx),
               'true_label':int(splits[h]['y_te'].iloc[idx]),
               'pred_prob':float(probs[idx]),
               'risk_label':'High' if probs[idx]>=stacking[h]['tau'] else 'Low'}
        if cf is not None:
            row.update({'cf_feature': cf['feature'],
                        'cf_old': cf['old_val'],
                        'cf_new': cf['new_val'],
                        'cf_pred_after': cf['p_new']})
        else:
            row.update({'cf_feature':None,'cf_old':None,'cf_new':None,'cf_pred_after':None})
        cf_records.append(row)
pd.DataFrame(cf_records).to_csv(f'{OUT_DIR}/csv/15_counterfactuals_top_highrisk.csv', index=False)
print('Saved 15_counterfactuals_top_highrisk.csv')


== Counterfactual explanations (horizon t0) ==

== Counterfactual explanations (horizon t1) ==

== Counterfactual explanations (horizon t2) ==
Saved 15_counterfactuals_top_highrisk.csv


## Section 10 — Multi-Task Heads: Dropout / GPA / Graduation / Risk-Level / Attendance

In [ ]:
# ============================================================
# CELL 10.1  -  Build multi-task targets
# ============================================================
import numpy as np
df_mt = df.copy()
# Graduation (1=graduate, 0=other)   (already Target==0 => Graduate)
df_mt['Graduation']   = (df_mt['Target']==0).astype(int)
# GPA — average grade across both semesters, normalised 0-4
def to_gpa(row):
    s1 = row['s1_grade']; s2 = row['s2_grade']
    n1 = row['s1_approved']; n2 = row['s2_approved']
    if n1 + n2 == 0:
        return 0.0
    avg = (s1*n1 + s2*n2) / max(1, n1+n2)
    # Portugal scale 0-20; convert to 4.0 scale
    gpa = (avg / 20.0) * 4.0
    return float(np.clip(gpa, 0, 4))
df_mt['GPA'] = df_mt.apply(to_gpa, axis=1).astype(float)
# Attendance (1 = daytime, 0 = evening)
df_mt['Attendance'] = df_mt['Attendance'].astype(int)
# Risk level — coarse-tier mapping
def risk_tier(p):
    if p < 0.30: return 'Low'
    if p < 0.60: return 'Medium'
    return 'High'

# attach per-horizon dropout probability predictions to build the Risk label
df_mt['Risk_t2'] = None
t2_probs = stacking['t2']['p_test_cal']
test_idx = splits['t2']['X_te'].index
for i, idx in enumerate(test_idx):
    df_mt.loc[idx, 'Risk_t2'] = risk_tier(t2_probs[i])

df_mt.to_csv(f'{OUT_DIR}/csv/16_multitask_labels.csv', index=False)
print('Multi-task target distributions:')
print('Graduation:', df_mt['Graduation'].value_counts().to_dict())
print('GPA: mean', round(df_mt['GPA'].mean(), 3), 'std', round(df_mt['GPA'].std(), 3))
print('Attendance:', df_mt['Attendance'].value_counts().to_dict())
print('Risk_t2 (test only):', df_mt['Risk_t2'].dropna().value_counts().to_dict())

Multi-task target distributions:
Graduation: {1: 2209, 0: 1421}
GPA: mean 2.125 std 0.993
Attendance: {1: 3222, 0: 408}
Risk_t2 (test only): {'Low': 394, 'High': 264, 'Medium': 68}


In [ ]:
# ============================================================
# CELL 10.2  -  Multi-task per-head metrics
# ============================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression

mt_metrics = []
for h in horizons:
    X_tr, X_te = splits[h]['X_tr'], splits[h]['X_te']
    y_tr, y_te = splits[h]['y_tr'], splits[h]['y_te']
    test_idx = X_te.index
    grad_true = df_mt.loc[test_idx, 'Graduation'].astype(int).to_numpy()
    gpa_true  = df_mt.loc[test_idx, 'GPA'].to_numpy()
    att_true  = df_mt.loc[test_idx, 'Attendance'].astype(int).to_numpy()
    risk_true = df_mt.loc[test_idx, 'Risk_t2'].map({'Low':0,'Medium':1,'High':2}).fillna(1).astype(int).to_numpy()
    grad_tr   = df_mt.loc[X_tr.index, 'Graduation'].astype(int).to_numpy()
    gpa_tr    = df_mt.loc[X_tr.index, 'GPA'].to_numpy()
    att_tr    = df_mt.loc[X_tr.index, 'Attendance'].astype(int).to_numpy()

    # Graduation head
    lr_g = LogisticRegression(max_iter=500, random_state=SEED).fit(X_tr, grad_tr)
    p_g  = lr_g.predict_proba(X_te)[:,1]
    auc_g= roc_auc_score(grad_true, p_g); acc_g = accuracy_score(grad_true, (p_g>=0.5).astype(int))
    # GPA regression head
    lr_rg = LinearRegression().fit(X_tr, gpa_tr)
    gpa_pred = np.clip(lr_rg.predict(X_te), 0, 4)
    mae_gpa = mean_absolute_error(gpa_true, gpa_pred); rmse_gpa = np.sqrt(mean_squared_error(gpa_true, gpa_pred)); r2_gpa = r2_score(gpa_true, gpa_pred)
    # Attendance head
    lr_a = LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced').fit(X_tr, att_tr)
    p_a  = lr_a.predict_proba(X_te)[:,1]
    auc_a= roc_auc_score(att_true, p_a); acc_a = accuracy_score(att_true, (p_a>=0.5).astype(int))
    # Risk-level classification using horizon's calibrated probs
    p_d = stacking[h]['p_test_cal']
    risk_pred_int = np.digitize(p_d, [0.30, 0.60])
    # use dropout itself as the proxy risk if t2, else use horizon's stacking
    if h=='t2':
        risk_acc = accuracy_score(risk_true, risk_pred_int)
    else:
        risk_acc = float('nan')
    mt_metrics.append({
        'horizon':h,
        'graduation_auc':float(auc_g), 'graduation_acc':float(acc_g),
        'gpa_mae':float(mae_gpa), 'gpa_rmse':float(rmse_gpa), 'gpa_r2':float(r2_gpa),
        'attendance_auc':float(auc_a), 'attendance_acc':float(acc_a),
        'risk_acc_t2_eq':float(risk_acc) if h=='t2' else None
    })

mt_df = pd.DataFrame(mt_metrics)
mt_df.to_csv(f'{OUT_DIR}/csv/17_multitask_metrics.csv', index=False)
print(mt_df.round(4))
print('Saved 17_multitask_metrics.csv')

  horizon  graduation_auc  graduation_acc  gpa_mae  gpa_rmse  gpa_r2  \
0      t0          0.8148          0.7576   0.6051    0.8857  0.2448   
1      t1          0.9371          0.8912   0.0998    0.2551  0.9373   
2      t2          0.9698          0.9435   0.0911    0.2088  0.9580   

   attendance_auc  attendance_acc  risk_acc_t2_eq  
0          0.9818          0.9353             NaN  
1          0.9347          0.8747             NaN  
2          0.9110          0.8471             1.0  
Saved 17_multitask_metrics.csv


## Section 11 — Top-Risk-Factor Impact (per horizon)

In [ ]:
# ============================================================
# CELL 11.1  -  Risk-Factor impact: global + per-student direction
# ============================================================
def risk_family(name):
    """Map a feature name to a high-level family for aggregated SHAP impact reporting."""
    n = name.lower()
    # Order matters: check more specific patterns first.
    if any(s in n for s in ['s1_to_s2', 'momentum', 'cum_', 'avg_grade',
                            'grade_per_approved', 'low_grade_flag_s2', 'zero_grade_s2',
                            's2_']):
        return '2nd-Semester'
    if any(s in n for s in ['s1_']):
        return '1st-Semester'
    if any(s in n for s in ['debtor','tuition','scholarship','financial',
                            'vulnerability','gdp','inflation','unemp']):
        return 'Financial/Macro'
    if any(s in n for s in ['admission','previousgrade','admission_vs','admissionx']):
        return 'Academic Preparation'
    if any(s in n for s in ['age','gender','marital','nationality','international','specialneeds']):
        return 'Demographics'
    if any(s in n for s in ['course','applicationmode','applicationorder','prevqual',
                            'motherqual','fatherqual','motherocc','fatherocc','parent','displaced']):
        return 'Course/Family'
    if any(s in n for s in ['academic_distress','total_failures','risk']):
        return 'Risk Composite'
    return 'Other'

risk_impact_dfs = []
for h in horizons:
    cols = splits[h]['X_tr'].columns
    feats = list(cols)
    families = [risk_family(c) for c in feats]
    imp = shap_results[h]
    sv = imp['shap_values']; Xte = splits[h]['X_te']
    fam_imp = {}
    for fam in set(families):
        idxs = [i for i,f in enumerate(families) if f==fam]
        if not idxs: continue
        fam_imp[fam] = float(np.abs(sv[:, idxs]).mean())
    fam_df = pd.DataFrame([{'horizon':h, 'family':k, 'mean_abs_shap':v} for k,v in fam_imp.items()])\
             .sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    fam_df.to_csv(f'{OUT_DIR}/csv/18_risk_family_impact_{h}.csv', index=False)
    risk_impact_dfs.append(fam_df)
    # feature-level risk factor table
    feat_df = pd.DataFrame({
        'horizon': h, 'feature': cols, 'family': families,
        'mean_abs_shap': np.abs(sv).mean(axis=0),
        'mean_shap': sv.mean(axis=0),
        'direction': np.where(sv.mean(axis=0)>0, 'Risk','Protective')
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    feat_df.to_csv(f'{OUT_DIR}/csv/19_risk_factor_per_feature_{h}.csv', index=False)

# Big concatenated table
all_feat = pd.concat([pd.read_csv(f'{OUT_DIR}/csv/19_risk_factor_per_feature_{h}.csv') for h in horizons], ignore_index=True)
all_feat.to_csv(f'{OUT_DIR}/csv/19_risk_factor_all_horizons.csv', index=False)
print('Saved family + per-feature risk-factor CSVs.')
for h in horizons:
    print('\n',h,'top 10 risk factors:')
    print(pd.read_csv(f'{OUT_DIR}/csv/19_risk_factor_per_feature_{h}.csv').head(10))

Saved family + per-feature risk-factor CSVs.

 t0 top 10 risk factors:
  horizon            feature                family  mean_abs_shap  mean_shap  \
0      t0    TuitionUpToDate       Financial/Macro       0.516254  -0.000450   
1      t0             Course         Course/Family       0.449404   0.039428   
2      t0        Scholarship       Financial/Macro       0.363441   0.033459   
3      t0                Age          Demographics       0.319987  -0.013140   
4      t0             Gender          Demographics       0.315192   0.049483   
5      t0     financial_risk       Financial/Macro       0.301782   0.035027   
6      t0  age_x_scholarship       Financial/Macro       0.209395   0.013251   
7      t0      PreviousGrade  Academic Preparation       0.204058   0.005660   
8      t0    ApplicationMode         Course/Family       0.182058  -0.002648   
9      t0        unemp_x_age       Financial/Macro       0.176696  -0.005273   

    direction  
0  Protective  
1        Risk  


In [ ]:
# ============================================================
# CELL 11.2  -  Bar charts of top risk factors per horizon
# ============================================================
fig_fam, axes_fam = plt.subplots(1, 3, figsize=(18,5))
for i, h in enumerate(horizons):
    fam_df = risk_impact_dfs[i]
    axes_fam[i].barh(fam_df['family'][::-1], fam_df['mean_abs_shap'][::-1], color=f'C{i}')
    axes_fam[i].set_title(f'Risk-Family Impact — {h}')
    axes_fam[i].set_xlabel('Mean |SHAP|')
    axes_fam[i].grid(alpha=0.3)
fig_fam.tight_layout()
fig_fam.savefig(f'{OUT_DIR}/figures/08_risk_family_impact.png', dpi=120, bbox_inches='tight')
plt.close(fig_fam)
print('Saved 08_risk_family_impact.png')

fig_topfeat, axes_tf = plt.subplots(3, 1, figsize=(10,15))
for i, h in enumerate(horizons):
    d = pd.read_csv(f'{OUT_DIR}/csv/19_risk_factor_per_feature_{h}.csv').head(12)
    axes_tf[i].barh(d['feature'][::-1], d['mean_abs_shap'][::-1], color=f'C{i+2}')
    axes_tf[i].set_title(f'Top-12 Risk Factors — {h}')
    axes_tf[i].set_xlabel('Mean |SHAP|')
    axes_tf[i].grid(alpha=0.3)
fig_topfeat.tight_layout()
fig_topfeat.savefig(f'{OUT_DIR}/figures/09_top_risk_factors_per_horizon.png', dpi=120, bbox_inches='tight')
plt.close(fig_topfeat)
print('Saved 09_top_risk_factors_per_horizon.png')

Saved 08_risk_family_impact.png
Saved 09_top_risk_factors_per_horizon.png


## Section 11b — From Explanations to Action: Three Research-Practice Questions

Section 11 established *which* features drive dropout risk. The three diagrams below turn
that into the three things a paper on an early-warning system actually needs to answer, and
none of them is a plain SHAP bar chart re-skinned:

1. **"What do advisors learn?"** (11b.1) — Individual SHAP values are too granular for an
   advisor to act on one student at a time. Instead, test-set students are clustered on their
   *family-level* SHAP signature (not raw features) into a small number of **risk archetypes**
   — recognisable "student profiles" (e.g. financially-strained-but-academically-fine vs.
   academically-disengaged-despite-no-financial-risk) that an advisor can learn to recognise
   and respond to differently, visualised as a radar "fingerprint" per archetype plus a 2-D
   map of the whole cohort.
2. **"What interventions become possible?"** (11b.2) — SHAP impact alone doesn't tell an
   institution what to *do*; a feature can be highly predictive and completely outside
   institutional control (age) or highly predictive and directly leverable (scholarship,
   tuition status). Every feature is scored on an **actionability axis** (institutional
   lever / supportable-via-programs / external-macro / fixed-demographic) and plotted against
   its SHAP impact, so the top-right quadrant is a ranked, ready-to-use intervention priority
   list rather than an importance ranking.
3. **"What institutional policies change?"** (11b.3) — Because this pipeline predicts at
   three horizons (t0/t1/t2), it can show *when* different risk families take over, not just
   which ones matter overall. The drift from financial/demographic dominance at t0 to
   academic-momentum dominance at t1/t2 is mapped directly onto a **timed policy-lever table**
   (e.g. "front-load scholarship disbursement before term start" vs. "trigger mandatory
   tutoring after week-6 grade check"), which is the kind of temporal-governance argument a
   single-horizon SHAP study can't make.


In [ ]:
# ============================================================
# CELL 11b.1  -  "What do advisors learn?"
#   Risk-Archetype Clustering: cluster test-set students on their
#   FAMILY-LEVEL SHAP signature (not raw features, not raw
#   probability) so the clusters are recognisable *reasons*, not
#   just risk tiers. Computed at horizon t2 (richest feature set =
#   the most complete picture an advisor would have by year-end);
#   the same code works unchanged at t0/t1 by changing ARCH_HORIZON.
# ============================================================
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler as _SSc

ARCH_HORIZON = 't2'
N_ARCHETYPES = 4

h = ARCH_HORIZON
sv = shap_results[h]['shap_values']                 # (n_test, n_features), signed SHAP
cols = shap_results[h]['columns']
families = [risk_family(c) for c in cols]
fam_names = sorted(set(families))

# Family-level signed SHAP signature per student (sum of signed SHAP within family)
fam_sig = np.zeros((sv.shape[0], len(fam_names)))
for j, fam in enumerate(fam_names):
    idxs = [i for i, f in enumerate(families) if f == fam]
    fam_sig[:, j] = sv[:, idxs].sum(axis=1)

p_test = stacking[h]['p_test_cal']
y_te   = splits[h]['y_te'].to_numpy()

# Cluster in standardised family-signature space (equal weight per family regardless of scale)
Z = _SSc().fit_transform(fam_sig)
km = KMeans(n_clusters=N_ARCHETYPES, random_state=SEED, n_init=10).fit(Z)
cluster_id = km.labels_

# Auto-label each archetype from its own centroid: top-2 families pushing risk UP,
# plus the cluster's mean predicted dropout probability.
archetype_rows = []
archetype_labels = {}
for c in range(N_ARCHETYPES):
    mask = cluster_id == c
    mean_fam = fam_sig[mask].mean(axis=0)
    order = np.argsort(-mean_fam)                 # most risk-increasing families first
    top2 = [fam_names[o] for o in order[:2] if mean_fam[o] > 0]
    if not top2:
        top2 = [fam_names[order[0]]]
    mean_risk = float(p_test[mask].mean())
    tier = 'High' if mean_risk >= 0.60 else ('Medium' if mean_risk >= 0.30 else 'Low')
    label = f'{tier}-Risk: ' + ' + '.join(top2)
    archetype_labels[c] = label
    archetype_rows.append({
        'archetype_id': c, 'label': label, 'n_students': int(mask.sum()),
        'pct_of_cohort': float(mask.mean() * 100), 'mean_dropout_prob': mean_risk,
        'true_dropout_rate': float(y_te[mask].mean()),
        **{f'mean_shap__{fam_names[j]}': float(mean_fam[j]) for j in range(len(fam_names))},
    })

df_archetypes = pd.DataFrame(archetype_rows).sort_values('mean_dropout_prob', ascending=False).reset_index(drop=True)
df_archetypes.to_csv(f'{OUT_DIR}/csv/23_advisor_risk_archetypes_{h}.csv', index=False)

df_student_archetype = pd.DataFrame({
    'test_index': np.arange(sv.shape[0]),
    'archetype_id': cluster_id,
    'archetype_label': [archetype_labels[c] for c in cluster_id],
    'dropout_probability': p_test, 'true_label': y_te,
})
df_student_archetype.to_csv(f'{OUT_DIR}/csv/23_student_archetype_assignments_{h}.csv', index=False)

print(f'Advisor risk archetypes ({h}):')
print(df_archetypes[['archetype_id','label','n_students','pct_of_cohort','mean_dropout_prob','true_dropout_rate']]
      .round(3).to_string(index=False))

# ---- Figure A: 2-D PCA map of the cohort, coloured by archetype ----
pca = PCA(n_components=2, random_state=SEED).fit(Z)
Z2 = pca.transform(Z)
fig = plt.figure(figsize=(15, 6))
ax_map = fig.add_subplot(1, 2, 1)
palette = plt.cm.tab10(np.linspace(0, 1, N_ARCHETYPES))
for c in range(N_ARCHETYPES):
    m = cluster_id == c
    ax_map.scatter(Z2[m, 0], Z2[m, 1], s=18, alpha=0.6, color=palette[c],
                    label=f'#{c}: {archetype_labels[c]}')
ax_map.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax_map.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax_map.set_title(f'Advisor Risk-Archetype Map — {h} ({horizon_desc[h]})')
ax_map.legend(fontsize=7, loc='best')
ax_map.grid(alpha=0.3)

# ---- Figure B: radar "fingerprint" — mean family-level SHAP per archetype ----
ax_radar = fig.add_subplot(1, 2, 2, projection='polar')
angles = np.linspace(0, 2*np.pi, len(fam_names), endpoint=False).tolist()
angles += angles[:1]
for c in range(N_ARCHETYPES):
    vals = df_archetypes.loc[df_archetypes['archetype_id'] == c,
                              [f'mean_shap__{f}' for f in fam_names]].values.flatten().tolist()
    vals += vals[:1]
    ax_radar.plot(angles, vals, linewidth=2, color=palette[c], label=f'#{c}')
    ax_radar.fill(angles, vals, alpha=0.08, color=palette[c])
ax_radar.set_xticks(angles[:-1]); ax_radar.set_xticklabels(fam_names, fontsize=8)
ax_radar.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax_radar.set_title('Archetype SHAP Fingerprint (mean signed SHAP per family)', fontsize=10)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/figures/17_advisor_risk_archetypes_{h}.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Saved 17_advisor_risk_archetypes_{h}.png, 23_advisor_risk_archetypes_{h}.csv, '
      f'23_student_archetype_assignments_{h}.csv')


Advisor risk archetypes (t2):
 archetype_id                                     label  n_students  pct_of_cohort  mean_dropout_prob  true_dropout_rate
            3  High-Risk: 2nd-Semester + Risk Composite         155         21.350              0.892              0.968
            0 High-Risk: 2nd-Semester + Financial/Macro         101         13.912              0.845              0.960
            2                           Low-Risk: Other         140         19.284              0.196              0.093
            1    Low-Risk: Academic Preparation + Other         330         45.455              0.192              0.073
Saved 17_advisor_risk_archetypes_t2.png, 23_advisor_risk_archetypes_t2.csv, 23_student_archetype_assignments_t2.csv


In [ ]:
# ============================================================
# CELL 11b.2  -  "What interventions become possible?"
#   Actionability x Impact Quadrant. SHAP impact tells us WHAT
#   predicts dropout; it says nothing about whether the institution
#   can DO anything about it. Every feature gets a curated
#   actionability score, then impact (x) vs. actionability (y) is
#   plotted so the top-right quadrant IS the intervention priority
#   list -- not an importance ranking re-labelled.
# ============================================================
ACTIONABILITY_SCORE = {
    'Institutional Lever':      3,   # institution can act on this directly & quickly
    'Supportable via Programs': 2,   # institution can influence indirectly (tutoring, advising)
    'External/Macro':           1,   # outside institutional control (economy-wide)
    'Fixed/Demographic':        0,   # immutable, or unethical to target directly
}

def feature_actionability(name):
    """Curated actionability taxonomy (documented for the paper's methodology section).
    Order matters: more specific patterns are checked first."""
    n = name.lower()
    # Direct financial/administrative levers the institution controls.
    if any(s in n for s in ['scholarship', 'tuitionuptodate', 'debtor', 'financial_risk',
                            'vulnerability_idx']):
        return 'Institutional Lever'
    # Academic-support levers: performance signals the institution can respond to with
    # tutoring, early-warning triggers, advising nudges, remedial sessions, etc.
    if any(s in n for s in ['s1_', 's2_', 'academic_distress', 'total_failures',
                            'momentum_score', 'cum_efficiency', 'grade_per_approved',
                            'eval_participation_gap', 'admission_vs_prevgrade']):
        return 'Supportable via Programs'
    # Macro-economic terms (including their interaction features) -- exogenous to the institution.
    if any(s in n for s in ['gdp', 'inflation', 'unemp']):
        return 'External/Macro'
    # Immutable student attributes / historical facts -- useful for early identification,
    # not appropriate or possible as an intervention target.
    return 'Fixed/Demographic'

quad_rows = []
for h in horizons:
    feat_df = pd.read_csv(f'{OUT_DIR}/csv/19_risk_factor_per_feature_{h}.csv')
    feat_df['actionability'] = feat_df['feature'].apply(feature_actionability)
    feat_df['actionability_score'] = feat_df['actionability'].map(ACTIONABILITY_SCORE)
    feat_df['horizon'] = h
    quad_rows.append(feat_df)
df_quad = pd.concat(quad_rows, ignore_index=True)
df_quad.to_csv(f'{OUT_DIR}/csv/24_intervention_actionability_matrix.csv', index=False)

# Priority list: only features that are actually leverable, ranked by impact, per horizon.
df_priority = (df_quad[df_quad['actionability_score'] >= 2]
               .sort_values(['horizon', 'mean_abs_shap'], ascending=[True, False]))
df_priority.to_csv(f'{OUT_DIR}/csv/24_intervention_priority_list.csv', index=False)
print('Top actionable interventions per horizon:')
for h in horizons:
    top = df_priority[df_priority['horizon'] == h].head(5)
    print(f'\n  {h}:')
    print(top[['feature', 'actionability', 'mean_abs_shap', 'direction']].round(4).to_string(index=False))

# ---- Quadrant figure, one panel per horizon ----
act_colors = {'Institutional Lever': '#2a9d8f', 'Supportable via Programs': '#e9c46a',
              'External/Macro': '#8d99ae', 'Fixed/Demographic': '#e76f51'}
rng = np.random.RandomState(SEED)
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
for i, h in enumerate(horizons):
    sub = df_quad[df_quad['horizon'] == h].copy()
    jitter = rng.uniform(-0.12, 0.12, size=len(sub))            # jitter the categorical y-axis
    for act, color in act_colors.items():
        m = sub['actionability'] == act
        axes[i].scatter(sub.loc[m, 'mean_abs_shap'], sub.loc[m, 'actionability_score'] + jitter[m],
                        s=40, alpha=0.75, color=color, label=act)
    # annotate the top-3 highest-impact LEVERABLE features (score >= 2)
    leverable = sub[sub['actionability_score'] >= 2].sort_values('mean_abs_shap', ascending=False).head(3)
    for _, r in leverable.iterrows():
        axes[i].annotate(r['feature'], (r['mean_abs_shap'], r['actionability_score']),
                         fontsize=7, xytext=(4, 4), textcoords='offset points')
    axes[i].axvline(sub['mean_abs_shap'].median(), color='grey', linestyle=':', linewidth=1)
    axes[i].set_yticks([0, 1, 2, 3])
    axes[i].set_yticklabels(['Fixed/\nDemographic', 'External/\nMacro',
                              'Supportable\nvia Programs', 'Institutional\nLever'], fontsize=8)
    axes[i].set_xlabel('Impact (mean |SHAP|)')
    axes[i].set_title(f'Intervention Quadrant — {h} ({horizon_desc[h]})')
    axes[i].grid(alpha=0.3, axis='x')
axes[0].set_ylabel('Actionability')
handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8, label=a)
           for a, c in act_colors.items()]
fig.legend(handles=handles, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.06), fontsize=9)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/figures/18_intervention_actionability_quadrant.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('\nSaved 18_intervention_actionability_quadrant.png, 24_intervention_actionability_matrix.csv, '
      '24_intervention_priority_list.csv')


Top actionable interventions per horizon:

  t0:
               feature            actionability  mean_abs_shap  direction
       TuitionUpToDate      Institutional Lever         0.5163 Protective
           Scholarship      Institutional Lever         0.3634       Risk
        financial_risk      Institutional Lever         0.3018       Risk
     age_x_scholarship      Institutional Lever         0.2094       Risk
admission_vs_prevgrade Supportable via Programs         0.1341 Protective

  t1:
           feature            actionability  mean_abs_shap  direction
  s1_approval_rate Supportable via Programs         0.9383       Risk
 academic_distress Supportable via Programs         0.7580 Protective
   TuitionUpToDate      Institutional Lever         0.2855       Risk
s1_pass_efficiency Supportable via Programs         0.2446       Risk
          s1_grade Supportable via Programs         0.2074       Risk

  t2:
           feature            actionability  mean_abs_shap  direction
   

In [ ]:
# ============================================================
# CELL 11b.3  -  "What institutional policies change?"
#   Policy-Lever Timeline: because this pipeline predicts at three
#   horizons, it can show not just WHICH risk family matters but
#   WHEN it takes over -- and pair each family with a concrete,
#   time-appropriate policy lever an institution could deploy at
#   that horizon, rather than a single static importance ranking.
# ============================================================
POLICY_LEVER = {
    'Financial/Macro':      'Front-load scholarship/aid decisions & payment-plan offers before term start (t0); flag debtors for proactive outreach.',
    'Academic Preparation': 'Bridge/remedial modules for students admitted with weak prior grades, assigned before classes begin (t0).',
    'Demographics':         'Targeted (not diagnostic) outreach for older / displaced / part-time-coded students during orientation (t0) -- context, not a control lever.',
    'Course/Family':        'Course-level advising capacity planning; first-generation / low-parental-education student mentoring programs (t0).',
    '1st-Semester':         'Mandatory early-semester (week 4-6) grade check-ins; automatic tutoring referral on low approval/evaluation rate (t1).',
    '2nd-Semester':         'Momentum-triggered intervention: automatic advising contact when s1->s2 grade or approval trend is negative (t2).',
    'Risk Composite':       'Composite risk-score dashboards for advisors, refreshed each horizon, to prioritise caseloads (t0-t2).',
    'Other':                '(unclassified features -- inspect individually before assigning a lever).',
}

fam_wide_rows = []
for i, h in enumerate(horizons):
    fam_df = risk_impact_dfs[i].copy()
    total = fam_df['mean_abs_shap'].sum()
    fam_df['horizon'] = h
    fam_df['share_of_total_impact'] = fam_df['mean_abs_shap'] / total
    fam_df['suggested_policy_lever'] = fam_df['family'].map(POLICY_LEVER).fillna(POLICY_LEVER['Other'])
    fam_wide_rows.append(fam_df)
df_policy = pd.concat(fam_wide_rows, ignore_index=True)
df_policy.to_csv(f'{OUT_DIR}/csv/25_policy_lever_timeline.csv', index=False)

print('Policy-relevant risk-family drift across horizons (share of total SHAP impact):')
pivot = df_policy.pivot_table(index='family', columns='horizon', values='share_of_total_impact').reindex(columns=horizons)
print((pivot * 100).round(1).to_string())

# ---- Figure: family share-of-impact drift across t0 -> t1 -> t2 ----
fig, (ax_line, ax_stack) = plt.subplots(1, 2, figsize=(16, 6))
fam_names_all = sorted(df_policy['family'].unique())
palette = plt.cm.tab10(np.linspace(0, 1, len(fam_names_all)))
color_map = dict(zip(fam_names_all, palette))

for fam in fam_names_all:
    ys = [pivot.loc[fam, h] * 100 if fam in pivot.index and not pd.isna(pivot.loc[fam, h]) else 0
          for h in horizons]
    ax_line.plot(horizons, ys, marker='o', linewidth=2, color=color_map[fam], label=fam)
ax_line.set_ylabel('% of total SHAP impact')
ax_line.set_title('Risk-Family Drift Across Horizons')
ax_line.legend(fontsize=8, loc='upper right')
ax_line.grid(alpha=0.3)

stack_data = np.nan_to_num(pivot.reindex(fam_names_all).to_numpy()) * 100
ax_stack.stackplot(horizons, stack_data, labels=fam_names_all, colors=[color_map[f] for f in fam_names_all], alpha=0.85)
ax_stack.set_ylabel('% of total SHAP impact (stacked)')
ax_stack.set_title('Composition of Risk Drivers by Horizon')
ax_stack.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/figures/19_policy_lever_timeline.png', dpi=120, bbox_inches='tight')
plt.close(fig)

# ---- Table figure: dominant family + lever per horizon, for a quick paper exhibit ----
dominant_rows = []
for h in horizons:
    sub = df_policy[df_policy['horizon'] == h].sort_values('share_of_total_impact', ascending=False)
    top = sub.iloc[0]
    dominant_rows.append({'horizon': h, 'description': horizon_desc[h],
                          'dominant_family': top['family'],
                          'share_of_impact_pct': round(top['share_of_total_impact']*100, 1),
                          'suggested_policy_lever': top['suggested_policy_lever']})
df_dominant = pd.DataFrame(dominant_rows)
df_dominant.to_csv(f'{OUT_DIR}/csv/25_dominant_family_per_horizon.csv', index=False)
print('\nDominant risk family & suggested policy lever per horizon:')
print(df_dominant[['horizon','dominant_family','share_of_impact_pct']].to_string(index=False))
for _, r in df_dominant.iterrows():
    print(f"\n  [{r['horizon']}] {r['dominant_family']} ({r['share_of_impact_pct']}%) -> {r['suggested_policy_lever']}")

print('\nSaved 19_policy_lever_timeline.png, 25_policy_lever_timeline.csv, 25_dominant_family_per_horizon.csv')


Policy-relevant risk-family drift across horizons (share of total SHAP impact):
horizon                 t0    t1    t2
family                                
1st-Semester           NaN  18.4   4.1
2nd-Semester           NaN   NaN  10.7
Academic Preparation  29.8   9.1  10.9
Course/Family         19.8   6.2   5.4
Demographics          21.0   4.2   1.6
Financial/Macro       25.2   8.9   7.4
Other                  4.3   0.5   0.4
Risk Composite         0.0  52.6  59.6

Dominant risk family & suggested policy lever per horizon:
horizon      dominant_family  share_of_impact_pct
     t0 Academic Preparation                 29.8
     t1       Risk Composite                 52.6
     t2       Risk Composite                 59.6

  [t0] Academic Preparation (29.8%) -> Bridge/remedial modules for students admitted with weak prior grades, assigned before classes begin (t0).

  [t1] Risk Composite (52.6%) -> Composite risk-score dashboards for advisors, refreshed each horizon, to prioritise caselo

## Section 12 — Per-Horizon Performance Comparison

In [ ]:
# ============================================================
# CELL 12.1  -  Per-horizon metric comparison
# ============================================================
metric_bars = ['accuracy','precision','recall','f1','auc']
fig, axes = plt.subplots(1, 2, figsize=(14,5))
x = np.arange(len(metric_bars))
width = 0.27
for i,h in enumerate(horizons):
    row = df_metrics[df_metrics['horizon']==h].iloc[0]
    axes[0].bar(x + i*width, [row[m] for m in metric_bars], width=width, label=h)
axes[0].set_xticks(x+width); axes[0].set_xticklabels(metric_bars)
axes[0].set_ylabel('Score'); axes[0].set_title('Per-horizon metric comparison')
axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')

# AUC deltas
aucs = [df_metrics[df_metrics['horizon']==h].iloc[0]['auc'] for h in horizons]
axes[1].plot(horizons, aucs, '-o', color='C3', linewidth=3, markersize=12)
axes[1].set_ylabel('AUC-ROC'); axes[1].set_title('AUC progression across horizons')
axes[1].grid(alpha=0.3)
for h,a in zip(horizons, aucs):
    axes[1].annotate(f'{a:.3f}', (h,a), textcoords='offset points', xytext=(0,10), ha='center')
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/figures/10_per_horizon_metrics.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Saved 10_per_horizon_metrics.png')

Saved 10_per_horizon_metrics.png


## Section 13 — At-Risk Student Identification (Output Table)

In [ ]:
# ============================================================
# CELL 13.1  -  Identify at-risk students at each horizon
# ============================================================
for h in horizons:
    s = stacking[h]
    tau = s['tau']
    p_test = s['p_test_cal']; y_te = splits[h]['y_te'].to_numpy()
    pred = (p_test >= tau).astype(int)
    risk_label = np.where(p_test >= 0.60, 'High', np.where(p_test >= 0.30, 'Medium', 'Low'))
    out = pd.DataFrame({
        'test_index': np.arange(len(y_te)),
        'true_label': y_te,
        'dropout_probability': p_test,
        'binary_prediction': pred,
        'risk_label': risk_label,
        'tau_used': tau,
    })
    out.to_csv(f'{OUT_DIR}/csv/20_atrisk_predictions_{h}.csv', index=False)

# Combined at-risk table
combined = []
test_idx = splits['t2']['X_te'].index
for h in horizons:
    s = stacking[h]; tau = s['tau']
    p_test = s['p_test_cal']; y_te = splits[h]['y_te'].to_numpy()
    pred = (p_test >= tau).astype(int)
    risk_label = np.where(p_test >= 0.60, 'High', np.where(p_test >= 0.30, 'Medium', 'Low'))
    out = pd.DataFrame({
        'horizon': h, 'test_index_local': np.arange(len(y_te)),
        'true_label': y_te, 'dropout_probability': p_test,
        'binary_prediction': pred, 'risk_label': risk_label,
    })
    combined.append(out)
combined_df = pd.concat(combined, ignore_index=True)
combined_df.to_csv(f'{OUT_DIR}/csv/20_atrisk_predictions_ALL_HORIZONS.csv', index=False)
print('Saved per-horizon and combined at-risk CSVs.')
print('Counts of at-risk students per horizon (binary prediction=1):')
print(combined_df.groupby('horizon')['binary_prediction'].sum())

Saved per-horizon and combined at-risk CSVs.
Counts of at-risk students per horizon (binary prediction=1):
horizon
t0    268
t1    298
t2    313
Name: binary_prediction, dtype: int64


In [ ]:
# ============================================================
# CELL 13.2  -  At-risk summary visualisation
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12,4))
counts = combined_df.groupby('horizon')['binary_prediction'].sum()
axes[0].bar(counts.index, counts.values, color=['C0','C1','C2'])
axes[0].set_title('At-risk students flagged per horizon')
axes[0].set_ylabel('# flagged as dropout')
axes[0].grid(alpha=0.3, axis='y')

rl_counts = combined_df.groupby(['horizon','risk_label']).size().unstack(fill_value=0)
rl_counts = rl_counts[['Low','Medium','High']]
rl_counts.plot(kind='bar', stacked=True, ax=axes[1], color=['#88c4a3','#f0c674','#e07b5b'])
axes[1].set_title('Risk-tier distribution per horizon')
axes[1].set_ylabel('# students')
axes[1].grid(alpha=0.3, axis='y')
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/figures/11_atrisk_counts.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Saved 11_atrisk_counts.png')

Saved 11_atrisk_counts.png


## Section 14 — Cross-Horizon Metric Comparison Table & Final Summary

In [ ]:
# ============================================================
# CELL 14.1  -  Final consolidated summary table
# ============================================================
summary = df_metrics[['horizon','description','accuracy','precision','recall','f1','auc','brier','logloss','tau','T']].round(4)
summary.columns = ['Horizon','Description','Accuracy','Precision','Recall','F1','AUC','Brier','LogLoss','τ* (Youden-J)','T* (Temp)']
print(summary.to_string(index=False))
summary.to_csv(f'{OUT_DIR}/csv/21_FINAL_summary_table.csv', index=False)
print('Saved 21_FINAL_summary_table.csv')

Horizon  Description  Accuracy  Precision  Recall     F1    AUC  Brier  LogLoss  τ* (Youden-J)  T* (Temp)
     t0 pre_semester    0.7851     0.7388  0.6972 0.7174 0.8545 0.1461   0.4507         0.4037        1.0
     t1   after_sem1    0.8871     0.8389  0.8803 0.8591 0.9469 0.0755   0.2591         0.3672        1.0
     t2   after_sem2    0.9160     0.8562  0.9437 0.8978 0.9736 0.0581   0.2368         0.3422        1.0
Saved 21_FINAL_summary_table.csv


## Section 15 — Lightweight Sanity Check: HistGradientBoosting baseline (for the comparative-analysis table)

ExtraTrees is now a full stacking base learner (Section 4/5), so it is not refit here. HistGradientBoosting is kept as a single independent sanity-check baseline — it is architecturally similar to LightGBM (histogram-binned gradient boosting) so it is not added as an eighth base learner, but it's cheap to compute and useful as an extra reference point in the paper's comparison table.

In [ ]:
# ============================================================
# CELL 15.1  -  HistGradientBoosting reference baseline at each horizon
#               (ExtraTrees numbers are read from the stacking base-learner
#               results already computed in Section 5 -- no need to refit it)
# ============================================================
from sklearn.ensemble import HistGradientBoostingClassifier
extra_rows = []
for h in horizons:
    Xtr, ytr = splits[h]['X_tr'], splits[h]['y_tr']
    Xte, yte = splits[h]['X_te'], splits[h]['y_te']
    hgb = HistGradientBoostingClassifier(max_iter=400, max_depth=10, learning_rate=0.05, random_state=SEED).fit(Xtr, ytr)
    p_hgb = hgb.predict_proba(Xte)[:,1]
    et_auc = float(roc_auc_score(yte, test_store[h]['ExtraTrees'])) if 'ExtraTrees' in test_store[h] else float('nan')
    extra_rows.append({'horizon':h,
                        'et_auc (stacking base learner)': et_auc,
                        'hgb_auc (reference only)': float(roc_auc_score(yte, p_hgb))})
extra_df = pd.DataFrame(extra_rows)
extra_df.to_csv(f'{OUT_DIR}/csv/22_baselines_et_hgb.csv', index=False)
print(extra_df.round(4))
print('Saved 22_baselines_et_hgb.csv')


  horizon  et_auc (stacking base learner)  hgb_auc (reference only)
0      t0                          0.8278                    0.8258
1      t1                          0.9440                    0.9449
2      t2                          0.9737                    0.9725
Saved 22_baselines_et_hgb.csv


## Section 16 — Package every CSV / PNG / model into a ZIP

In [ ]:
# ============================================================
# CELL 16.1  -  ZIP everything for download
# ============================================================
import zipfile, glob
zip_path = f'{OUT_DIR}_RESULTS.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(f'{OUT_DIR}/**/*', recursive=True):
        if os.path.isfile(f):
            arc = os.path.relpath(f, start='.')
            zf.write(f, arc)
print(f'Zipped to {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)')

# download via google.colab if available
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Not running in Colab; please download the ZIP manually.')

Zipped to ./DEEP_SEF_OUTPUT_RESULTS.zip (3875.8 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# CELL 16.2  -  Final inventory of files generated
# ============================================================
print(f'\n{"="*70}\nINVENTORY — {OUT_DIR}\n{"="*70}')
for root, dirs, files in os.walk(OUT_DIR):
    for fn in sorted(files):
        full = os.path.join(root, fn)
        print(f'  {full.replace(OUT_DIR+"/",""):<70}  {os.path.getsize(full)/1024:7.1f} KB')
print('\nDone — notebook finished.')


INVENTORY — ./DEEP_SEF_OUTPUT
  meta/06_hpo_best_params_t0.json                                             1.2 KB
  meta/06_hpo_best_params_t1.json                                             1.2 KB
  meta/06_hpo_best_params_t2.json                                             1.2 KB
  meta/conformal_preds_t0.npy                                                11.5 KB
  meta/conformal_preds_t1.npy                                                11.5 KB
  meta/conformal_preds_t2.npy                                                11.5 KB
  models/stacker_t0.joblib                                                   31.1 KB
  models/stacker_t1.joblib                                                  117.0 KB
  models/stacker_t2.joblib                                                  230.9 KB
  figures/01_confusion_matrices.png                                          39.5 KB
  figures/02_roc_overlay_horizons.png                                        61.7 KB
  figures/03_reliability_diagrams.

## Appendix A — Reading Guide for the Research Paper
Below is a quick mapping between the artefacts produced and the sections/figures you can reference in the paper.

| Notebook artefact | Description | Paper section |
|---|---|---|
| `csv/21_FINAL_summary_table.csv` | Per-horizon accuracy / F1 / AUC / τ* / T* | §5.5 Early-Signal Performance |
| `csv/11_stacking_metrics.csv` | Detailed metrics per horizon | §5.3 Final Results |
| `csv/06_hpo_results_*.csv` | Optuna best-CV-AUC per base learner | §4.4 Bayesian HPO |
| `csv/09_oof_probs_*.csv` | Out-of-fold probabilities | §4.6 Stacking |
| `csv/18_risk_family_impact_*.csv` | Family-level SHAP impact | §8.1 Interpretability / Risk-Factor Analysis |
| `csv/19_risk_factor_per_feature_*.csv` | Per-feature SHAP — top risk drivers per horizon | §5.5 / §8.1 |
| `csv/15_counterfactuals_top_highrisk.csv` | Counterfactuals for high-risk students | §5.6 Counterfactual XAI |
| `csv/20_atrisk_predictions_*.csv` | Per-horizon at-risk student flag table | §5.5 |
| `csv/17_multitask_metrics.csv` | Graduation / GPA / Attendance / Risk head metrics | §4.8 Multi-task |
| `figures/01_confusion_matrices.png` | t0 / t1 / t2 confusion matrices | §5.4 |
| `figures/02_roc_overlay_horizons.png` | ROC overlay (t0/t1/t2) | §5.4 |
| `figures/03_reliability_diagrams.png` | Calibration reliability per horizon | §4.7 Isotonic / Temperature Scaling |
| `figures/04_open_world_rejection.png` | OOD score distribution | §4.7 Open-World Rejection |
| `figures/05_shap_summary_*.png` | SHAP beeswarm per horizon | §4.8 SHAP |
| `figures/06_shap_bar_*.png` | Top-15 SHAP bar per horizon | §4.8 |
| `figures/07_shap_dependence_*.png` | Dependence plots for top-3 features | §4.8 |
| `figures/08_risk_family_impact.png` | Family-level SHAP impact | §8.1 |
| `figures/09_top_risk_factors_per_horizon.png` | Top risk-factor features per horizon | §5.5 |
| `figures/10_per_horizon_metrics.png` | Per-horizon metric comparison | §5.5 |
| `figures/11_atrisk_counts.png` | At-risk bar + tier distribution | §5.5 |

### New artefacts added in this update (base-learner diagnostics, meta-learner selection, ensemble validation)

| Notebook artefact | Description | Paper section |
|---|---|---|
| `csv/09b_base_learner_metrics.csv` | Accuracy/Precision/Recall/F1/AUC for every individual base learner (incl. MLP, NaiveBayes) per horizon | §5.2 Base-Learner Performance |
| `figures/12_base_learner_accuracy.png` | Bar chart of per-horizon base-learner accuracy | §5.2 Base-Learner Performance |
| `figures/13_base_learner_all_metrics.png` | All metrics, averaged across horizons, per base learner | §5.2 Base-Learner Performance |
| `csv/10b_meta_learner_comparison.csv` | LightGBM-meta vs CatBoost-meta CV-AUC, and which was selected, per horizon | §5.4 Meta-Learner Selection |
| `figures/14_meta_learner_comparison.png` | Bar chart comparing candidate meta-learners | §5.4 Meta-Learner Selection |
| `csv/12_stacking_vs_baselines.csv` | Stacking AUC vs best single base learner vs simple mean blend | §5.4 Ensemble Validation |
| `figures/15_base_learner_correlation.png` | Correlation heatmap of base-learner predictions (diversity check) | §5.4 Ensemble Validation |
| `figures/16_stacking_vs_baselines.png` | Bar chart: does stacking actually beat simpler baselines? | §5.4 Ensemble Validation |

**Note on base learners (superseded — see below):** an earlier version of this pipeline included an ANN (`MLPClassifier`) and Gaussian Naive Bayes; `09b_base_learner_metrics.csv` and `12_stacking_vs_baselines.csv` are exactly the diagnostics that showed they weren't earning their place, which is why they were removed in favour of ExtraTrees (see the note below).

### New artefacts added in this update (leaner base-learner set + TARB backbone + research-question diagrams)

| Notebook artefact | Description | Paper section |
|---|---|---|
| `csv/10c_tarb_vs_incumbent.csv` | TARB nested-CV AUC vs incumbent LightGBM/CatBoost meta-learner, per horizon, and which backbone was adopted | §5.4 Ensemble Backbone Selection |
| `figures/16b_tarb_vs_incumbent.png` | Bar chart: TARB vs incumbent meta-learner CV-AUC (winner labeled) | §5.4 Ensemble Backbone Selection |
| `csv/22_baselines_et_hgb.csv` | ExtraTrees (stacking base learner) vs HistGradientBoosting (reference-only) test AUC | §5.2 Base-Learner Performance |
| `csv/23_advisor_risk_archetypes_t2.csv` | Auto-labelled student risk archetypes (family-level SHAP clustering) with cohort share & mean risk | §6 "What do advisors learn?" |
| `csv/23_student_archetype_assignments_t2.csv` | Per-student archetype assignment | §6 "What do advisors learn?" |
| `figures/17_advisor_risk_archetypes_t2.png` | Cohort PCA map + radar SHAP fingerprint per archetype | §6 "What do advisors learn?" |
| `csv/24_intervention_actionability_matrix.csv` | Every feature scored on Impact (SHAP) x Actionability (institutional lever / supportable / macro / fixed) | §6 "What interventions become possible?" |
| `csv/24_intervention_priority_list.csv` | Ranked, leverable-only intervention shortlist per horizon | §6 "What interventions become possible?" |
| `figures/18_intervention_actionability_quadrant.png` | Impact x Actionability quadrant plot, per horizon | §6 "What interventions become possible?" |
| `csv/25_policy_lever_timeline.csv` | Risk-family share of SHAP impact per horizon + suggested policy lever | §6 "What institutional policies change?" |
| `csv/25_dominant_family_per_horizon.csv` | Dominant risk family & recommended lever, one row per horizon | §6 "What institutional policies change?" |
| `figures/19_policy_lever_timeline.png` | Line + stacked-area chart of risk-family drift across t0/t1/t2 | §6 "What institutional policies change?" |

**Note on base learners (revised):** the MLP (ANN) and Gaussian Naive Bayes base learners were removed after diagnostics (`09b_base_learner_metrics.csv` / `12_stacking_vs_baselines.csv`) showed neither ever beat the weakest tree model on this dataset. **ExtraTrees** replaces them: it is reported as the strongest individual model in head-to-head benchmarks on this exact UCI dataset in the literature, and its randomised-split structure is genuinely decorrelated from the gradient-boosted trees already in the ensemble (diversity without an accuracy penalty — see Section 4 markdown for citations). The base-learner set is now LightGBM, CatBoost, XGBoost, ExtraTrees, FT-Transformer, TabPFN.

**Note on the ensembling backbone:** Section 6.1c adds **TARB** (TabPFN-Anchored Residual Boosting) as a second candidate backbone alongside the LightGBM/CatBoost stacking meta-learner. TARB only replaces the incumbent for a horizon if it beats it on nested CV-AUC (see `10c_tarb_vs_incumbent.csv`), so every downstream section automatically uses whichever backbone actually won for that horizon.
